Script per la generazione di una mappa di bordo per il tuo itinerario di viaggio

In [ ]:
import folium
from folium import DivIcon
# from folium.plugins import MiniMap
import requests
import time

# 1. Database completo delle 22 tappe principali
tappe = [
    # FASE 1 (Giorni 1-7)
    ("Victoria Falls (Zambia/Zimbabwe)", [-17.9243, 25.8572], "🚁", "green"),
    ("Chobe National Park", [-17.8177, 25.1489], "🐘", "green"),
    ("Elephant Sands (Nata)", [-19.7486, 26.0683], "⛺", "green"),
    ("Maun", [-19.9833, 23.4167], "🛩️", "green"),
    ("Moremi Game (Okavango)", [-19.4298, 23.6288], "🦛", "green"),

    # FASE 2 (Giorni 8-14)
    ("Mohembo Border", [-18.2667, 21.7833], "🛂", "orange"),
    ("Rundu", [-17.9333, 19.7667], "🐊", "orange"),
    ("Grootfontein", [-19.5667, 18.1167], "🛒", "orange"),
    ("Namutoni (Etosha Est)", [-18.8055, 16.9859], "🦒", "orange"),
    ("Halali (Etosha Centro)", [-19.0354, 16.4714], "🦏", "orange"),
    ("Okaukuejo (Etosha Sud)", [-19.1812, 15.9178], "🦁", "orange"),
    ("Opuwo (Villaggio Himba)", [-18.0609, 13.8394], "🛖", "orange"),
    ("Epupa Falls", [-17.0008, 13.2458], "🌴", "orange"),
    ("Cape Cross (Otarie)", [-21.7667, 13.9833], "🦭", "orange"),

    # FASE 3 (Giorni 15-23)
    ("Spitzkoppe", [-21.8250, 15.1833], "⛰️", "darkred"),
    ("Swakopmund", [-22.6833, 14.5333], "🥨", "darkred"),
    ("Walvis Bay (Sandwich Harbour)", [-22.9559, 14.5073], "🦩", "darkred"),
    ("Solitaire", [-23.8833, 16.0000], "🥧", "darkred"),
    ("Sesriem / Sossusvlei", [-24.4833, 15.8000], "🏜️", "darkred"),
    ("Fish River Canyon", [-27.6186, 17.6083], "🌄", "darkred"),
    ("Cape Town (Table Mountain)", [-33.9249, 18.4241], "🚠", "darkred"),
    ("Cape of Good Hope", [-34.3568, 18.4740], "🐧", "darkred")
]

lats = [coord[1][0] for coord in tappe]
lons = [coord[1][1] for coord in tappe]

# Array per memorizzare i km percorsi per arrivare a ogni tappa
km_per_tappa = [0] * len(tappe)

# 2. Inizializzazione mappa con stile Atlante / National Geographic (Rimosso Zoom Control)
mappa_tour = folium.Map(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/NatGeo_World_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri National Geographic',
    control_scale=True,
    zoom_snap=0.1,
    zoom_delta=0.1,
    zoom_control=False
)

# 3. Inquadratura ristretta (Margine oceanico extra sotto Cape Town)
mappa_tour.fit_bounds([
    [min(lats) - 1.5, min(lons) - 0.5],
    [-12.0, max(lons) + 1.5]
])

# --- FUNZIONE CREAZIONE MARKER ---
def crea_marker_html(simbolo, colore_sfondo):
    if simbolo.isascii() and len(simbolo) > 2:
        contenuto = f"<i class='fa fa-{simbolo}' style='color: white;'></i>"
    else:
        contenuto = simbolo

    return f"""
    <div style="
        background-color: {colore_sfondo}; color: white; border-radius: 50%;
        width: 26px; height: 26px; display: flex; align-items: center; justify-content: center;
        border: 2px solid white; box-shadow: 0 2px 5px rgba(0,0,0,0.5); font-size: 13px;
        font-family: 'Apple Color Emoji', 'Segoe UI Emoji', 'Noto Color Emoji', sans-serif;
    ">
        {contenuto}
    </div>
    """

# --- MOTORE ANTI-SOVRAPPOSIZIONE ---
icon_posizionate = []

def calcola_coordinate_libere(lat, lon, soglia_minima=0.35, spostamento=0.45):
    nuova_lat, nuova_lon = lat, lon
    for _ in range(10):
        conflitto = False
        for p_lat, p_lon in icon_posizionate:
            distanza = ((nuova_lat - p_lat)**2 + (nuova_lon - p_lon)**2)**0.5
            if distanza < soglia_minima:
                nuova_lon += spostamento
                conflitto = True
                break
        if not conflitto:
            break

    icon_posizionate.append((nuova_lat, nuova_lon))
    return [nuova_lat, nuova_lon]

# 4. TRACCIAMENTO VOLO JOHANNESBURG - VICTORIA FALLS
coord_jnb_reale = [-26.1392, 28.2460]
coord_vfa_reale = [-18.0958, 25.8390]

folium.PolyLine(
    [coord_jnb_reale, coord_vfa_reale],
    color="#1E88E5", weight=3, dash_array='8, 8', opacity=0.8
).add_to(mappa_tour)

coord_jnb_marker = calcola_coordinate_libere(coord_jnb_reale[0], coord_jnb_reale[1])
coord_vfa_marker = calcola_coordinate_libere(coord_vfa_reale[0], coord_vfa_reale[1])

folium.Marker(
    location=coord_jnb_marker, tooltip="Volo: Johannesburg O.R. Tambo",
    icon=DivIcon(html=crea_marker_html("✈️", "#1E88E5"), icon_anchor=(13, 13))
).add_to(mappa_tour)

folium.Marker(
    location=coord_vfa_marker, tooltip="Volo: Arrivo a Victoria Falls",
    icon=DivIcon(html=crea_marker_html("✈️", "#1E88E5"), icon_anchor=(13, 13))
).add_to(mappa_tour)

# 5. Funzione OSRM (Ora restituisce anche i chilometri)
def ottieni_strada_e_distanza(start, end):
    url = f"http://router.project-osrm.org/route/v1/driving/{start[1]},{start[0]};{end[1]},{end[0]}?overview=full&geometries=geojson"
    try:
        r = requests.get(url)
        data = r.json()
        if data.get("code") == "Ok":
            coords = data["routes"][0]["geometry"]["coordinates"]
            distanza_m = data["routes"][0]["distance"]
            return coords, round(distanza_m / 1000) # Converte metri in KM
    except: pass
    return None, 0

# 6. Tracciamento dell'itinerario e calcolo chilometri
print("Calcolo il percorso e le distanze sulle strade reali...")
for i in range(len(tappe) - 1):
    start_data, end_data = tappe[i], tappe[i+1]

    if i < 4: color_linea = "#2E7D32"
    elif i < 13: color_linea = "#D97706"
    else: color_linea = "#B91C1C"

    percorso, dist_km = ottieni_strada_e_distanza(start_data[1], end_data[1])

    if percorso:
        folium.PolyLine([[lat, lon] for lon, lat in percorso], color=color_linea, weight=5, opacity=0.9).add_to(mappa_tour)
        km_per_tappa[i+1] = dist_km # Salva i chilometri per la tappa di destinazione
    else:
        folium.PolyLine([start_data[1], end_data[1]], color=color_linea, weight=5, dash_array='10').add_to(mappa_tour)

    time.sleep(0.2)

# 7. Posizionamento Marker Tappe
for i, (nome, coord, simbolo, colore) in enumerate(tappe):
    if colore == "green": col_hex = "#2E7D32"
    elif colore == "orange": col_hex = "#D97706"
    else: col_hex = "#B91C1C"

    coord_sicura = calcola_coordinate_libere(coord[0], coord[1])

    folium.Marker(
        location=coord_sicura, tooltip=f"Tappa {i+1}: {nome}",
        icon=DivIcon(html=crea_marker_html(simbolo, col_hex), icon_anchor=(13, 13))
    ).add_to(mappa_tour)

# 8. Generazione della leggenda con i Chilometri
def genera_lista_fase(start, end):
    lista_html = []
    for i in range(start, end):
        nome = tappe[i][0]
        simbolo = tappe[i][2]
        colore = tappe[i][3]
        chilometri = km_per_tappa[i]

        if colore == "green": col_hex = "#2E7D32"
        elif colore == "orange": col_hex = "#D97706"
        else: col_hex = "#B91C1C"

        if simbolo.isascii() and len(simbolo) > 2:
            icona_legenda = f"<i class='fa fa-{simbolo}' style='color: {col_hex}; width: 16px; text-align: center; margin-right: 3px; font-size: 11px;'></i>"
        else:
            icona_legenda = f"<span style='width: 16px; display: inline-block; text-align: center; margin-right: 3px; font-size: 13px; font-family: \"Apple Color Emoji\", \"Segoe UI Emoji\", \"Noto Color Emoji\", sans-serif;'>{simbolo}</span>"

        # Testo dinamico per i chilometri
        testo_km = f"<span style='color: #8c6d53; font-weight: normal; font-size: 9.5px; margin-left: 4px;'>(+{chilometri} km)</span>" if chilometri > 0 else ""

        lista_html.append(f"<li style='margin-bottom: 2px; font-size: 11px; display: flex; align-items: center;'>{icona_legenda} <b>{i+1}.</b>&nbsp;{nome}{testo_km}</li>")

        # Sottotitoli intermedi
        stile_sub = "font-size: 9.5px; color: #b45309; margin-left: 20px; margin-bottom: 3px; font-style: italic; font-family: Arial, sans-serif;"

        if i == 2: lista_html.append(f"<div style='{stile_sub}'>↪️ Sosta ai grandi baobab di Planet Baobab 🌳</div>")
        elif i == 4: lista_html.append(f"<div style='{stile_sub}'>↪️ Esplorazione del Delta dell'Okavango 🛶</div>")
        elif i == 11: lista_html.append(f"<div style='{stile_sub}'>↪️ Discesa costiera passando per Palmwag 🌴</div>")
        elif i == 14: lista_html.append(f"<div style='{stile_sub}'>↪️ Visita ai glifi di Bushman's Paradise 🎨</div>")
        elif i == 17: lista_html.append(f"<div style='{stile_sub}'>↪️ Attraversando il Tropico del Capricorno 🌐</div>")
        elif i == 19: lista_html.append(f"<div style='{stile_sub}'>↪️ Visita alla città fantasma di Kolmanskop 👻</div>")

    return "".join(lista_html)

# VECCHIA Minimappa in basso a destra
# minimap = MiniMap(tile_layer='CartoDB Positron', position='bottomright', width=160, height=160, zoom_level_offset=-5)
# mappa_tour.add_child(minimap)


# 9. Layout HTML Finale
html_layout = f"""
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.min.css">

<style>
    body {{ display: flex !important; flex-direction: column !important; margin: 0; height: 100vh; overflow: hidden; background: #f5f0e6; }}

    .legenda-container {{
        order: 1 !important; width: 100% !important; flex: 0 0 auto !important;
        background-color: #f5f0e6; color: #3d2c1e; padding: 6px 12px; box-sizing: border-box;
        font-family: 'Helvetica Neue', Arial, sans-serif;
        border-bottom: 4px solid #b45309; box-shadow: 0 2px 6px rgba(0,0,0,0.15);
    }}
    .legenda-header {{ text-align: center; margin-bottom: 2px; }}
    .legenda-container h1 {{ color: #2c1810; font-size: 13.5px; margin: 0; line-height: 1.1; display: inline-block; }}
    .legenda-container h2 {{ color: #8c6d53; font-size: 9.5px; margin: 0 0 0 6px; display: inline-block; font-weight: normal; }}

    .flags-banner {{
        display: flex; justify-content: center; gap: 20px; margin: 6px 0 8px 0; padding: 6px 10px;
        background: #ede4d3; border-radius: 4px; border: 1px solid #d4c5a9;
    }}
    .flag-item {{ display: flex; flex-direction: column; align-items: center; gap: 3px; }}
    .flag-item img {{
        height: 18px; width: 28px; object-fit: cover; border-radius: 2px;
        box-shadow: 0 1px 2px rgba(0,0,0,0.2); border: 1px solid #bfae92;
    }}
    .flag-item span {{ font-size: 8px; color: #5c4033; font-weight: bold; text-transform: uppercase; letter-spacing: 0.5px; }}

    .fasi-wrapper {{ display: flex; gap: 8px; justify-content: space-between; }}
    .fase-colonna {{ flex: 1; background: #fffdfa; padding: 4px 6px; border-radius: 4px; border: 1px solid #d4c5a9; }}
    .fase-titolo {{ font-size: 10px; padding: 2px; color: white; margin-bottom: 3px; border-radius: 3px; font-weight: bold; text-align: center; }}

    .fase-1 {{ background-color: #2E7D32; }}
    .fase-2 {{ background-color: #D97706; }}
    .fase-3 {{ background-color: #B91C1C; }}

    .legenda-container ul {{ list-style-type: none; padding: 0; line-height: 1.02; margin: 0; }}

    .folium-map {{ order: 2 !important; width: 100% !important; flex: 1 !important; position: relative !important; }}

    .bussola-css {{
        position: absolute; top: 15px; right: 15px; z-index: 1000;
        width: 35px; height: 35px; background: rgba(255, 253, 250, 0.95);
        border-radius: 50%; border: 2px solid #5c4033; box-shadow: 0 3px 6px rgba(0,0,0,0.3);
        display: flex; flex-direction: column; align-items: center; justify-content: center; pointer-events: none;
    }}
    .bussola-freccia {{ width: 0; height: 0; border-left: 3px solid transparent; border-right: 3px solid transparent; border-bottom: 8px solid #B91C1C; margin-bottom: 1px; }}
    .bussola-n {{ color: #5c4033; font-weight: bold; font-family: sans-serif; font-size: 11px; margin-top: 1px; line-height: 1; }}

    .minimappa-avm {{
        position: absolute; bottom: 25px; right: 25px; z-index: 1000;
        width: 140px; height: 140px; background: #fff;
        border-radius: 50%; border: 4px solid #d4c5a9;
        box-shadow: 0 4px 10px rgba(0,0,0,0.4);
        pointer-events: none; overflow: hidden;
    }}
    .minimappa-avm svg {{ width: 100%; height: 100%; display: block; }}
    .minimappa-avm path {{ fill: #2c2c2c; stroke: #ffffff; stroke-width: 1px; }}

    #locationMap-ZA, #locationMap-NA, #locationMap-BW, #locationMap-ZW, #locationMap-ZM {{
        fill: #008C45 !important;
    }}
</style>

<div class="legenda-container">
    <div class="legenda-header">
        <h1>Africa Australe Ovest</h1>
        <h2>Itinerario di 23 giorni</h2>
    </div>

    <div class="flags-banner">
        <div class="flag-item">
            <img src="https://flagcdn.com/w80/zm.png" title="Zambia" alt="Zambia">
            <span>Zambia</span>
        </div>
        <div class="flag-item">
            <img src="https://flagcdn.com/w80/zw.png" title="Zimbabwe" alt="Zimbabwe">
            <span>Zimbabwe</span>
        </div>
        <div class="flag-item">
            <img src="https://flagcdn.com/w80/bw.png" title="Botswana" alt="Botswana">
            <span>Botswana</span>
        </div>
        <div class="flag-item">
            <img src="https://flagcdn.com/w80/na.png" title="Namibia" alt="Namibia">
            <span>Namibia</span>
        </div>
        <div class="flag-item">
            <img src="https://flagcdn.com/w80/za.png" title="South Africa" alt="South Africa">
            <span>Sudafrica</span>
        </div>
    </div>

    <div class="fasi-wrapper">
        <div class="fase-colonna">
            <div class="fase-titolo fase-1">Fase 1 (Giorni 1-7): Cascate (Zambia/Zimbabwe) & Botswana</div>
            <ul>{genera_lista_fase(0, 5)}</ul>
        </div>
        <div class="fase-colonna">
            <div class="fase-titolo fase-2">Fase 2 (Giorni 8-14): Namibia & Etosha</div>
            <ul>{genera_lista_fase(5, 14)}</ul>
        </div>
        <div class="fase-colonna">
            <div class="fase-titolo fase-3">Fase 3 (Giorni 15-23): Deserto & Sudafrica</div>
            <ul>{genera_lista_fase(14, len(tappe))}</ul>
        </div>
    </div>
</div>

<div class="bussola-css" id="bussola-nord">
    <div class="bussola-freccia"></div><div class="bussola-n">N</div>
</div>

<!-- CONTENITORE DELLA NUOVA MINIMAPPA -->
<div class="minimappa-avm" id="minimappa-custom">

    <svg viewBox="0 0 768.8 768.8"><circle cx="384.4" cy="384.4" r="384.4" fill="#fff"></circle><path id="locationMap-AO" d="m331.9 454.78c-2.1081.348-5.6636.5155-7.5723 1.336.9477 1.5487 1.0129 2.8708 2.5371 6.0527 1.219-.7089 4.725-3.0729 5.7676-4.7969-.4076-1.7246-.1075-1.3758-.7324-2.5918zm23.918 9.1875c-2.0224-.029-4.0572.1268-5.8809.3047-6.7742-.8651-15.342 1.6768-22.037 1.457 1.2243 4.687 8.2866 14.727 6.4668 19.648-1.9136 5.5669.052 10.792 2.9199 15.545 1.193 3.8142 3.8218 7.6544.4004 11.137 2.1177 5.4166-.918 9.6087-5.7852 11.799-2.7119 4.376-5.6695 8.7578-7.0097 13.84-1.5785 8.733-4.206 16.345-2.4961 24.25 3.4529-.751 7.7521-3.1639 12.715-1.7871s10.231 2.3745 15.5 2.2246c8.5729.07 17.352-1.4532 25.772.6641 4.6776 2.4326 9.9864 2.7014 15.117 3.291 6.3179 1.8553 19.726 1.2317 27.592-.029-2.9415-2.7501-8.0536-11.815-12.215-13.518-.4837-5.0929-.038-10.285-.062-15.418.5544-3.8737-.8285-8.6744 1.0352-11.953 4.7094-.1 9.4339.08 14.15.025 3.7343-.958 1.2906-11.768 3.3652-17.928-4.7684-.83-13.088 1.4357-14.865-7.7735s1.9092-13.329-3.9004-19.398c-5.4173-3.9814-16.606-9.2052-21.07-.4238-5.5535 4.2093-17.361 1.7453-17.93-6.377.2338-7.7527-5.7151-9.4941-11.781-9.5801z"></path><path id="locationMap-BF" d="m226.8 287.86c0 2e-4-6.4111.3372-9.4707.502-2.9269.6535-4.6807 3.6699-7.7988 3.9707-3.2734.4614-3.3723 4.1289-5.8281 5.6523-2.1819 1.2703-3.8931 3.0893-5.1973 5.2227-2.003 2.4136-5.6894 1.3678-7.8809 3.3457-1.7952 2.28-3.5334 4.6769-4.8261 7.2734-.3386 2.6667-1.234 5.1106-3.2539 6.9551-.6481 1.7021-2.8513 5.1264-2.2657 6.336 1.2299-.069 2.2461-.1109 3.3184 1.0625 1.1669 1.3751 2.8151 5.3592 5.7168 4.9531s4.5578-1.7194 7.043-.8379 3.778-.7632 5.3965-.9727c.757-2.0943-1.6304-5.9902-.9629-8.541 1.8803-2.3221 5.2372-1.0588 7.8007-1.3984 4.0773-.3709 8.17-.9199 12.262-.3633 3.3504.036 5.5248.7725 8.832 1.25 1.3778.073 2.5095.039 3.707-.334-.035-1.074 1.7302-3.6461 3.5274-3.7441 2.4254-.2357 4.918.3325 7.3105-.2461 2.5039-.7463 2.463-3.2119 2.8321-5.1426-7e-4-7e-4.0.0.0.0-.27-.4919-1.3101.6405-2.8438-.6836s-5.1017-4.4391-7.9003-5.7129-1.6326-4.726-4.168-6.3848-5.0184-5.7241-5.2793-8.0605-.068-4.0997-.068-4.0997z"></path><path id="locationMap-BI" d="m479.7 435.11c-.3158.5302-1.6704-.07-2.0957.4902-1.3319 1.1502-2.9693 2.9102-4.6817 2.2656-1.7124-.6447-.4482-1.5005-3.5839-1.8613.3826 2.0902 1.6142 4.4062 1.6992 6.5527.015 1.3098-.5282 2.547-1.2735 3.5977-1.0742 1.8383-1.6311 5.0487-1.5605 7.25.6363-1.586 5.1421-1.5834 7.0898-3.2149 6.0395-6.027 9.3067-7.4265 4.4063-15.08z"></path><path id="locationMap-BJ" d="m255.3 311.11c-2.07.5364-6.2471.5318-8.2539 1.6876-.3695 1.931-.3274 4.398-2.8321 5.1445-2.3925.5786-4.8851.01-7.3105.2461-1.7972.098-3.562 2.6701-3.5274 3.7441 1.5158.9563 3.0985 2.4693 3.295 4.25 1.0006 3.962 2.9195 7.8103 2.5371 12.004-.045 3.2844-.041 6.5711-.1055 9.8555-.135 2.6239.8479 5.0814 1.3398 7.6133-.1293 2.6358-.9799 6.8272-.9785 7.873 2.2849.1093 6.8996-.9054 9.0098-1.6816-.2196-2.3345.7811-4.0776.3594-7.4102.3202-4.9414-.4877-10.224 1.6777-14.844 3.143-4.2526 5.7209-9.122 6.9316-14.332 1.4028-4.3649-2.4649-9.8911-2.1425-14.15z"></path><path id="locationMap-BW" d="m432.5 565.68c.5225 3.5722-10.818 6.3394-10.582 2.7383-5.204.7086-18.612.3163-23.553 2.084-4.0961 4.0083-2.2725 10.482-3.207 15.674-.9506 4.4909 1.5726 12.474-2.6582 13.871s-8.0487-.1143-6.9863 5.0273c-.6265 7.0439-.4237 14.36-.6055 21.422 3.6697.353 8.5201 2.5396 7.7676 6.7696.1404 4.3027-.052 8.6821.5371 12.932 3.0732 1.2747 7.8318 2.3579 10.006-1.0703 3.7768-3.4891 5.7777-8.5389 9.7442-11.803 3.7044-.711 5.7529 4.8371 10.113 3.5059 3.4188-.7096 8.0803 1.1418 10.422-2.1387 2.6968-2.6744.8016-7.5262 3.9239-9.7383 2.9515.2634 5.4262-.3701 6.5273-3.6054 2.4792-4.7517 6.7693-8.1992 10.488-11.951 2.8525-2.1743 8.3498-2.0412 9.3566-6.5618-8.3733.1257-11.343-10.28-13.54-10.807-3.1571-2.4422-2.835-5.634-7.6602-8.8965s-2.8071-10.97-10.094-17.451z" class="highlighted"></path><path id="locationMap-CD" d="m454.8 369.21c-8.3503.7009-22.597 1.6044-30.676 3.2695-4.335 1.0678-10.16.2135-9.9492 6.3496-7.4501 5.104-13.694-2.7293-21.451-3.25-3.5969-4.9922-5.8263-1.2888-10.986 1.1875-2.9762 1.4283-2.7778 5.4653-2.2598 9.4316-.049 1.6793-.1966 3.1095-.3535 5.6934s-3.5324 6.1664-3.6074 9.7949c-.1008 4.753.084 9.674-1.9434 14.117-1.4128 4.7243-6.1954 6.969-9.123 10.58-2.8933 2.7065-1.4676 7.39-3.7266 10.096-3.1234 1.1097-2.1168 4.6361-2.5742 7.1504-1.2032 3.2118-5.4758 3.8484-7.625 6.4492-2.11 1.8032-3.4279 5.0843-6.8027 4.4141-2.9969-.1391-6.0301.7739-8.9766.1074-1.3548-.8621-1.3688-.029-2.8203.2109.6249 1.216.3248.8672.7324 2.5918-1.0426 1.724-4.5486 4.088-5.7676 4.7969.024.6242.7207 2.5859 1.0352 3.5605 6.6949.2198 15.263-2.3221 22.037-1.457 7.2947-.7115 17.974-1.0615 17.662 9.2754.5683 8.1223 12.376 10.586 17.93 6.377 4.4641-8.7814 15.653-3.5576 21.07.4238 5.8096 6.0691 2.1228 10.189 3.9004 19.398s10.097 6.9435 14.865 7.7735c6.5216-1.2527 12.148 10.245 19.299 9.7871 5.1715-1.3457 10 1.1664 13.742 4.5722 3.2524 3.1483 6.5354 7.3567 11.674 6.3457 4.2531-.8241 4.6171-8.2643 1.5195-10.578-3.6493 1.5241-8.6278 1.3222-10.709-2.9922-4.7352-3.3702-1.811-8.2287 1.1231-11.469.7414-3.3562-.3318-6.8811.1699-10.312 1.1699-4.2131 5.3402-7.1876 9.7148-5.7442 3.6618 1.3581 8.8366-2.0981 9.5235-2.5273-1.0781-1.5067-1.5783-3.6386-2.4785-5.2461-1.5653-4.0643-2.7951-8.2394-4.9942-12.059-1.9284-3.4521-1.977-7.6089-4.1133-10.924-1.3321-1.2891-1.9022-2.3538-1.6445-2.9961-.071-2.2013.4863-5.4117 1.5605-7.25.7453-1.0507 1.2885-2.2879 1.2735-3.5977-.085-2.1465-1.3166-4.4625-1.6992-6.5527-.4006-3.1265-.5002-6.1162.4101-9.0957.249-1.7535 3.6088-3.6197 3.9844-5.4824.1456-2.7206-.9708-5.4958-.037-8.1954.4849-2.4438 2.2069-4.2531 3.4394-6.3124.6009-2.2828 1.4909-4.4851 3.4141-6.0098 1.9203-1.9913 5.042-2.9077 5.9629-5.7344 1.4423-2.3057-1.5414-4.3926-.377-6.7637.6621-2.9319-3.1375-4.6873-3.8672-7.0586-8.6086-8.1355-24.239-.4515-27.48-12.146z"></path><path id="locationMap-CF" d="m412.8 321.06c-3.2996.1288-6.9474 2.6032-8.539 5.4023-2.2466 3.8785-5.0267 7.4416-8.668 10.07-3.9027 2.6987-9.9908.5722-12.988 4.4375-.3763 4.6796-5.2511 5.4793-8.8867 6.5508-4.0689 2.4459-12.305 1.803-16.838 2.502-2.3813 3.304-3.0445 4.8696-4.0879 8.7597-4.3198 6.7695-3.2941 13.056-.7265 17.74 2.5677 4.6842 8.5824 10.309 10.475 14.41 1.6243-3.4909 4.932-7.8378 9.2382-6.8359 3.4194.204 6.3496 1.9934 7.668 2.1054-.518-3.9663-.7164-8.0033 2.2598-9.4316 5.16-2.4763 7.3894-6.1797 10.986-1.1875 7.7575.5207 14.001 8.354 21.451 3.25-.2107-6.1361 5.6142-5.2818 9.9492-6.3496 8.0792-1.6651 22.326-2.5686 30.676-3.2695-2.4134-3.2664-4.1112-8.2159-7.9551-10.301-3.3683-2.583-3.3953-8.0889-8.1211-9.2305-5.3161-2.6052-7.703-7.6107-11.604-10.654-2.6841-2.0646-7.1592-3.0716-3.9961-6.5449 2.5024-4.7374-4.4501-9.282-8.0938-11.113-.6947-.245-1.4377-.3403-2.1992-.3105z"></path><path id="locationMap-CG" d="m370.2 383.96c-3.5401.2085-6.2506 3.9181-7.6718 6.9726-1.94 2.2118-3.7088 5.5479-5.4356 7.9121-3.7849-1.3226-7.8528-1.6613-11.986-1.1894-3.8502.066-7.5202-.7321-11.334-.7891-1.8345 2.0315.7914 7.5877 4.5078 7.8164 2.74.6044 5.1258-2.3516 7.9102-1.0918 3.9569 2.2401-.1657 6.2539-2.6758 7.6113-3.577 2.0032.5779 5.4122 2.8672 6.4317 2.2942 2.2775 1.6964 5.8822 2.3105 8.7148-1.4345 2.9327-.1419 7.0096-2.3789 9.3203-2.4013 1.7443-5.5229 2.3725-8.332 1.2149-3.5272-.1378-3.4823-5.1052-6.4004-5.8008-3.3948.7288-.6858 5.3362-4.3008 5.8008-1.9064-1.0955-5.3706.8047-2.9746 2.9238 2.04 1.7367 4.9548 4.8518 2.0527 7.2949s-3.4378-1.1625-5.4765-1.6191-2.4965.9008-3.7051 1.9297c1.3234 2.091 6.3701 7.2079 7.1322 8.7373 1.9087-.8205 5.4643-.9869 7.5724-1.3349 1.4515-.2396 1.466-1.0731 2.8208-.211 2.9465.6665 5.9797-.2465 8.9766-.1074 3.3748.6702 4.6927-2.6109 6.8027-4.4141 2.1492-2.6008 6.4218-3.2374 7.625-6.4492.4574-2.5143-.5492-6.0407 2.5742-7.1504 2.259-2.7057.8333-7.3892 3.7266-10.096 2.9276-3.6111 7.7102-5.8558 9.123-10.58 2.0275-4.4432 1.8426-9.3642 1.9434-14.117.075-3.6285 3.4505-7.211 3.6074-9.7949s.3043-4.0141.3535-5.6934c-1.3184-.112-4.2486-1.9014-7.668-2.1054-.5382-.1253-1.0606-.1665-1.5664-.1367z"></path><path id="locationMap-CI" d="m165.5 326.93c-2.3097.0-4.5612.2741-6.6347 1.2031.6778 3.8446 2.9722 8.0585 2.3535 12.041-.692 4.738-6.2268 6.5986-5.9981 11.594.036 2.1054-.5852 4.4956.1465 6.5117 1.524 3.8112 5.9118 5.1612 8.0254 8.5235 1.4047 2.1352.6764 4.7665.053 7.0332-.7123 1.9365.4512 3.7776.8223 5.7089 4.5866-.3351 8.5369-4.0464 12.117-4.9726 6.088-1.9314 12.362-3.2939 18.674-4.25 2.9718-.566 6.4239.8179 9.2344 1.584-.3462-2.0564.2181-2.491-.5723-4.7285.028-3.1106-2.2582-5.5029-3.2988-8.2813-.8674-2.4571 1.3826-4.6292.9883-7.1133-.2939-3.1868 2.3864-5.4117 3.414-8.1699.8706-3.3624-3.3252-8.9624-3.0527-12.289-1.6185.2095-2.9113 1.8542-5.3965.9727s-4.1412.4318-7.0429.8379-4.5499-3.578-5.7168-4.9531c-1.0723-1.1734-2.0885-1.1313-3.3184-1.0625-4.3501 1.1929-9.711-.1804-14.793-.1895z"></path><path id="locationMap-CM" d="m340.7 302.37c-1.0678 3.0051 3.1216 8.2534 3.0781 12.15-.806 3.8494-6.4407 4.9937-6.1446 9.3809 1.4584 3.9171-2.7033 6.1282-4.0312 9.2812-1.0972 4.2048-3.4684 7.7444-5.9648 11.273-2.4828 4.6417-3.8674 10.358-8.4102 13.502-4.4111 1.4702-8.892-3.3677-13.213-.6621-4.4346 2.3166-6.3865 7.4884-6.4336 12.254-.4659 2.5002-2.9177 4.0599-3.8164 6.5547 1.779 1.1423 5.418 6.4325 8.875 5.291-.2523 2.4503 3.5472 2.9238 1.8984 5.4199 2.8244 2.3493 1.2367 7.5528 1.5098 9.6719 1.1695.075 7.7743 1.0846 9.9688 1.25 1.0632.071 2.6726-.224 3.7363-.168 4.1404-.069 8.1247-.7502 12.041-.7031 3.8138.057 7.4838.8556 11.334.7891 4.1335-.4719 8.2014-.1332 11.986 1.1894 1.7268-2.3642 3.4956-5.7003 5.4356-7.9121-1.8922-4.101-7.9069-9.7259-10.475-14.41-2.5676-4.6841-3.5933-10.971.7265-17.74 1.0434-3.8901 1.7066-5.4557 4.0879-8.7597-3.2586-8.9085-7.9387-9.9631-15.051-14.861-4.0173-4.0464 2.8583-6.9987 6.377-6.7656 5.6983-.5324 2.6481-7.0589 1.1836-10.215-2.0781-3.3663 4.8158-6.91-.5508-8.8282-3.7315-.282-4.7637-7.1385-8.1484-6.9824zm-41.016 82.312c-.7067.021-1.6076.2103-2.1816.5-2.2275 1.5794.5364 4.0244 1.6738 1.125 2.322-1.2266 1.6857-1.6601.5078-1.625z"></path><path id="locationMap-CV" d="m29.6 279.78c.072.5927-.7776 1.7164.4117 1.6268 1.2813-.1914 1.1571-1.7965.024-2.1496-.4416-.2682-.588.1612-.4356.5228zm15.7 5.6c-.4686.441-.7986 2.2923.275 1.5125.3908-1.0324 2.0024-.6311 2.0552-1.7509-.3845-.8035-1.8321-.2355-2.3302.2384zm-8.6 9.1c.075.7606.6009 1.6536.024 2.4703-.5787.6036-.049 2.4576.7974 1.708.3003-.9592.7244-1.9286.7268-2.9488-.025-.7531-.6712-2.3234-1.5482-1.2295z"></path><path id="locationMap-DJ" d="m582.1 318.78c.458-.9237 1.2256-1.6214 2.0579-2.2027-.3443-.7222-1.2683-1.9376-2.103-1.9777-1.4731-.2174-2.5344-1.9472-1.9411-3.3344.5957-1.0088 1.9909-1.0579 2.6074-2.0584.5489-.8173 1.1148-1.7198 1.1787-2.7268-.025-1.4331-1.1761-2.4691-2.0404-3.4841-.6293-.6998-1.7757-1.4781-2.6557-1.8618-.6076 1.554-.9893 3.0528-2.1211 4.2834-.8766 1.1216-1.2274.6427-2.388 1.4575-1.2081.9571-3.2983 3.4646-3.5484 5.0832-.3227 1.7044-.4451 3.5633.2762 5.186.5642 1.1683 1.9741 1.45 3.1394 1.6177 1.1333.1645 2.2525-.113 3.3534-.3416 1.2824-.2013 2.8886.3575 4.1847.3597z"></path><path id="locationMap-DZ" d="m284.8 91.16c-7.2119-.008-13.366 1.8167-19.424 2.0449-6.8363-4.255-15.204 2.0313-22.978 1.1836-7.4221-1.9861-15.524.52398-20.475 6.2539-4.5201.97509-8.5702 3.7663-11.611 6.6797.094 3.4629.01 5.9061 1.2226 9.3457 2.9009 6.8044 3.7857 13.573 4.8887 20.512-3.9375 7.1152-18.978-3.2972-18.602 9.0996-1.8917 4.7203-9.1224 5.6254-11.273 11.096-5.6698 5.6071-14.499 1.394-20.494 6.7676-7.899.0437-8.0973 7.8306-8.332 14.037-.035.9121-.05 1.7872-.1113 2.6016 9.1223 7.0173 19.806 12.852 29.453 19.744 10.968 7.7978 22.826 17.066 33.658 25.055 5.1384 4.3996 12.008 7.2277 15.9 12.801 2.9766 6.6669 12.899 2.732 16 9.6992-1.9339 6.3619 9.3594 4.4668 9.3594 4.4668 6.0772-.015 13.598-3.4631 18.104-7.2031 13.094-9.0806 28.839-21.596 40.807-32.873 1.1095-6.5882-11.697-5.2376-12.76-10.438-1.5136-5.7554-8.5138-9.5806-4.1309-16.287-.4926-11.003.1848-23.269-.8457-34.197-1.0627-2.3751-2.5536-5.7911-3.7988-8.0742-.2828-2.2756-.9641-4.5387-2.4239-6.3555-1.1678-1.9235-3.1712-3.1688-4.2753-5.1113-.6953-2.6172-3.3065-3.8023-4.8731-5.7754-1.0897-2.4051.066-5.1984-.8711-7.6992.1706-2.5826 3.2195-3.1256 4.6309-4.8437 2.1669-2.1413.3986-5.0037-.668-7.1445.023-3.6418-3.5326-5.9988-3.8144-9.5488-1.1157-2.7851.2864-3.3657-.8067-5.8086-.4905-.0178-.9743-.0268-1.4551-.0273z"></path><path id="locationMap-EG" d="m469 135.25c-.6352-27e-5-1.3085.22154-2.0371.81836-3.3036-.43051-5.8471.61202-7.3926 3.6348-3.0502.78261-4.6409 5.8533-8.373 2.7598-4.2786-.0222-8.5198.54819-12.008-2.3594-4.1325.11772-10.321-2.2206-14.424-1.1133 1.3344 27.727.5359 54.867 2.3379 85.771 19.232-.3755 39.568-1.4884 58.791-2.0918 4.8823.079 10.433-.5178 14.266 3.2012 3.754-.037 2.7664-7.7545 6.8047-9.334.9718-2.4709 6.5816-2.3924 7.7832-3.5039-.1839-2.7732-4.4605-5.9187-.834-7.8535-2.8356-3.619-6.3925-6.8311-9.0254-10.688-3.6761-2.9139-5.897-7.0829-8.7754-10.713-1.516-3.7299-2.7225-7.4242-6.2383-10.012-4.2741-2.5754-3.5157-8.0188-6.0976-11.58-1.0101-2.6486-4.6764-5.1906-4.3536-7.5879 2.454-1.114 1.1581-7.689 3.6895-3.0195 2.1896 4.3374 3.2908 8.9748 6.3008 13 2.7131 3.8412 3.6855 8.6602 6.289 12.625 3.1217 4.7819 4.8198-1.1763 2.9356-3.9863-1.2392-4.1472.5995-7.7266.2187-11.594 2.4325-2.7677 2.2618-8.8358 4.2696-10.367l-8.2852-15.484c-1.3198 3.8934-7.0826 6.6919-11.029 5.207-3.5104-.7272-6.4403-2.6332-9.5996-3.9004-1.7387.16561-3.3074-1.8293-5.2129-1.8301z"></path><path id="locationMap-EH" d="m128 178.13c-3.2989-.01-6.5887.01-9.8574.047.0 6e-4-.01.0-.01.0-1.4272 2.9115-2.6415 6.3138-4.4942 8.9434-3.1336.5624-4.3505 3.5622-6.4355 5.5156-2.5689 2.0367-1.5414 5.7269-2.0469 8.543-.975 3.1295-5.0203 3.8373-6.8008 6.5488-2.5731 2.0307-3.7722 5.3847-3.8437 8.5488.5732 3.7566-3.6866 4.851-4.2891 8.1367-1.0368 3.2178-4.774 5.3054-5.5117 8.696 2.2674.9311 6.7066-.2804 9.7305.1126 7.5879.3007 15.175.7795 22.766.9336 3.2277-.1516 1.9385-4.0759 2.1387-6.1426.071-3.2558.055-6.8996 2.0195-9.6562 1.7845-2.2586 7.0181-1.6006 6.3457-5.3984.062-5.4671-.131-10.948.1133-16.406.6706-3.8243 5.2707-2.4235 7.998-2.6426 6.6903.085 13.404.403 20.084.057 2.344-1.9545.9503-5.9515 1.5215-8.7793.068-1.6501.1053-2.758.1719-4.4082.028-.6866.084-1.9151.1113-2.6016-9.8126.1252-19.806-.031-29.709-.047z"></path><path id="locationMap-ER" d="m540 258.47c-2.3496 2.7961-3.7815 6.3754-6.7813 8.5566-3.2917.5725-3.2489 3.9179-3.4004 6.5508.089 5.908-2.6133 9.6727-3.2031 15.551 2.9962 1.6433 7.1603.6936 10.146-.8028 2.8233-1.3736 5.7387.8424 8.6582.2481 2.562-.271 5.8794.2815 6.8262 3.0566.5922 2.6106 3.7491 1.8417 5.6582 2.6641 3.4884 1.1153 6.5529 3.4021 8.6308 6.4238 1.6522 2.014 4.9325 1.456 6.1758 3.9199.6479.7521 1.0513 1.863 2.0078 2.2344 1.1606-.8148 1.5121-.3354 2.3887-1.457 1.1318-1.2306 1.5135-2.7292 2.1211-4.2832-1.9838-.681-5.7193-2.5531-7.8477-2.668-.9148-2.4311-2.5575-4.1719-4.1562-6.0449-1.3239-1.7741-2.9884-3.4434-4.6719-4.7852-2.6357.7268-3.2685-1.5913-3.6836-3.5449-1.525-1.1142-4.0144-.4689-4.6465-2.9004-1.8623-1.973-1.3126 3.0332-2.9551.6953-1.5282-1.3045-3.5906-2.205-2.748-4.707-.7636-2.371-.7392-4.958-1.6602-7.2949-1.0586-2.331-1.2104-5.149-3.1894-7-1.0831-1.1744-2.3536-3.0422-3.6699-4.4121z"></path><path id="locationMap-ET" d="m538.9 287.87c-.716-.01-1.427.1136-2.1328.457-2.9862 1.4964-7.1503 2.4461-10.146.8028.153.6393-.044 1.4229.7617 3.0781 2.4709 5.7021-1.7892 10.398-5.8203 14.275s-2.3159 11.445-6.1113 13.914-5.9356 5.734-5.5157 8.3339c1.1066 4.0111 2.2901 3.3712.127 7.9473-.7263 5.0823-7.4113 3.1995-9.2813 7.1582-1.1268 5.9869 6.742 4.1304 9.3438 7.4922 2.7876 2.7289 5.4087 5.7142 6.3223 9.5918 2.5801 3.3014 6.7556 6.7114 7.1269 11.24 6.4783-1.1224 12.644 2.9033 17.398 6.8711 3.2982 4.2826 9.2632 6.8037 14.391 4.2715 6.2898-3.3049 14.098-5.8917 20.91-2.5156 2.9952-2.091 7.8784-1.994 9.752-5.7676 2.9258-4.4044 9.3784-1.0806 12.941-4.7187 6.5749-8.2335 13.97-15.766 21.211-23.395 5.1932-2.7128.2249-5.3927-3.33-3.7773-5.1043.1381-9.6091-3.0654-14.275-4.8516-4.3062-1.7328-8.7658-3.513-12.547-6.1992-1.1906-2.9052-8.9517-9.6637-7.8965-13.301-1.2961.0-2.9012-.5607-4.1836-.3594-1.1009.2286-2.2202.5063-3.3535.3418-1.1653-.1677-2.5745-.4488-3.1387-1.6171-.7213-1.6227-.6-3.4812-.2773-5.1856.2501-1.6186 2.3407-4.1269 3.5488-5.084-.9565-.3714-1.3599-1.4823-2.0078-2.2344-1.2433-2.4639-4.5236-1.9059-6.1758-3.9199-2.0779-3.0217-5.1424-5.3085-8.6308-6.4238-1.9091-.8224-5.066-.053-5.6582-2.6641-.9468-2.7751-4.2642-3.3276-6.8262-3.0566-2.1896.4457-4.3775-.6892-6.5254-.7051z"></path><path id="locationMap-GA" d="m333.8 396.86c-3.9163-.047-7.9006.6341-12.041.7031-.8769 2.7963 1.2954 7.8387-2.4277 8.8828-5.4437-.6375-11.303-1.233-14.662-.6152.067 2.9259-1.5266 7.262-1.5605 10.047-1.144 2.1412-1.2162 5.7852-3.5918 6.7129-.7161-1.0933-3.2061-3.5733-2.4082-.5137-.2651 2.9135 3.9199 4.2519 3.8008 6.3574-3.1495-1.3876.2905 2.8146.9844 3.8145 1.9633 2.6739 2.1225 6.7154 5.4511 8.2343 2.986.9955 7.4094 5.1534 9.8594 6.9239 1.2086-1.0289 1.6664-2.3863 3.7051-1.9297s2.5744 4.0622 5.4765 1.6191-.013-5.5582-2.0527-7.2949c-2.396-2.1191 1.0682-4.0193 2.9746-2.9238 3.615-.4646.906-5.072 4.3008-5.8008 2.7357.6521 2.8672 5.0589 5.7793 5.7207.1941.044.4006.072.6211.08 2.8091 1.1576 5.9307.5294 8.332-1.2149 2.237-2.3107.9444-6.3876 2.3789-9.3203-.6141-2.8326-.016-6.4373-2.3105-8.7148-2.2893-1.0195-6.4442-4.4285-2.8672-6.4317 2.5101-1.3574 6.6327-5.3712 2.6758-7.6113-2.2624-1.0236-4.2606.7358-6.4004 1.1074-.1646.029-.3316.048-.4981.059-.3329.021-.6692.0-1.0117-.074-3.7164-.2287-6.3423-5.7849-4.5078-7.8164z"></path><path id="locationMap-GH" d="m217.8 320.79c-3.0684-.063-6.1354.3156-9.1934.5938-2.5635.3396-5.9204-.9237-7.8007 1.3984-.6673 2.5501 1.7179 6.4444.9629 8.5391-3e-4 6e-4 2e-4.0.0.0 3e-4 3e-4.0-3e-4.0.0-.2725 3.3267 3.9233 8.9267 3.0527 12.289-1.0276 2.7582-3.7079 4.9831-3.414 8.1699.3943 2.4841-1.8557 4.6562-.9883 7.1133 1.0406 2.7784 3.3273 5.1708 3.2988 8.2813.7904 2.2375.2261 2.6721.5723 4.7285 2.0947 1.2558 7.4706 3.1702 9.7832 2.1738 1.9738-1.0255 3.3035-2.9799 5.416-3.8086 2.241-1.3232 5.0896-.6847 7.2481-2.2285 1.3672-1.1757 5.2256-1.9793 5.6113-2.2188-.1842-2.7209-2.4122-5.5196-2.3203-8.2773-.1521-2.4719.2445-4.9436 1.0273-7.2793.3484-3.165-1.6812-6.2218-.6484-9.3828.7716-2.4529-.9164-4.7054-.957-7.1406-.2247-2.1575.2149-4.3486-.3809-6.4727-.2177-1.5653.4384-3.4475.6289-5-3.3072-.4775-5.4816-1.214-8.832-1.25-1.023-.1391-2.0456-.2095-3.0684-.2305z"></path><path id="locationMap-GM" d="m102.6 297.54c-2.4089.013-4.8647.5366-7.2442.6328-2.2276-.3665-4.2416-.1267-6.4902.2539.8366.7952.1503 2.1216-.5293 2.5801-.6871.653-.5335 2.4994-.3516 3.4297.4756-.3054 1.9409.7866 2.7559.7656 2.448.4435 4.8893-.051 7.248-.7051 2.7063-.8454 5.441.1238 8.17.2188 2.0846.2168 5.3507-.3399 5.2148-3.0313-.6886-1.9349-2.9067-2.4373-4.5137-3.414-1.3855-.5625-2.8144-.7383-4.2597-.7305z"></path><path id="locationMap-GN" d="m113.8 307.79c.076 1.1024-1.3962 2.0758-2.5742 3.6094-.2709.9324-.1188 1.9313-.4277 2.8613-.1916.882-.9364 1.4925-1.7715 1.7461-1.6417.5904-3.4726.6201-5.0078 1.5195-1.1271.6897-2.2472 2.0305-3.3164 2.8008.1909 2.8177 3.9193 3.1351 5.4375 4.9629 1.6488 2.159 3.3528 4.278 4.541 6.7324 1.3493 2.9943 3.3892 5.7151 4.5195 8.1856 2.5207-1.7388 4.811-3.6968 7.0352-5.7871 1.394-.9077 3.0351-1.318 4.6172-1.7559 1.5002-.6865 3.2237-1.3264 4.8613-.7285 1.0727.6841 2.3001 1.152 3.3203 1.9023.6378.9612 1.2584 1.9495 1.7813 2.9844.7071 1.2389.5792 2.6958.2636 4.0293-.3876 1.3905-.035 3.0685 1.2969 3.8242.7023.6275 2.3993.3334 3.6602 1.082 7.9773-4.8699 5.049 6.5055 13.125 6.0039-.2287-4.9952 5.306-6.8557 5.998-11.594.6187-3.9825-1.6757-8.1964-2.3535-12.041-3e-4-4e-4 3e-4.0.0.0-.5648-2.7644-2.1475-6.0363-4.0195-8.2285-1.722-2.0526-2.5078-4.6076-3.3028-7.1055-1.8189-2.8244-4.7191-.03-6.5312 1.2852-1.9677.9904-4.2342 1.1627-6.3008 1.9004-1.9417.6262-3.2289-1.1991-4.8379-2.1719-4.0267-1.7339-8.8541.3866-11.688-1.4668-2.3439-1.8267-4.9602-4.5058-8.3262-4.5488z"></path><path id="locationMap-GQ" d="m308 396.48c-.01 1.013-.1941 3.0629-1.0273 3.7773-.7676.737-1.5891 1.5125-1.8711 2.5782-.2823.7853-.4775 2.152-.4864 2.9941 3.3589-.6178 9.2185-.022 14.662.6152 3.7231-1.0441 1.5508-6.0865 2.4277-8.8828-1.0637-.056-2.6731.239-3.7363.168-2.1945-.1654-8.7993-1.1747-9.9688-1.25z"></path><path id="locationMap-GW" d="m105.4 307.52c-.9375-.013-1.8789-.011-2.8281.018-3.7441.036-6.9077 2.1227-10.713 3.623.0.0 2.3243.5401 3.3515.4922.9859-.047 1.2471-.3555 2.2247-.1386.994.4254-.5772.9495-1.0157 1.0898-.7433.084-1.4881.8773-.5703 1.6484.9368-.095 1.9394-.4424 2.8418.022 1.0198.5897.8323 1.9206 1.0371 2.9121.2477.9037.8196 2.2165 1.0129 3.1383 1.0692-.7703 2.1901-2.1107 3.3172-2.8004 1.5352-.8994 3.3661-.9291 5.0078-1.5195.8351-.2536 1.5799-.8641 1.7715-1.7461.3089-.93.1568-1.9289.4277-2.8613 1.178-1.5336 2.6504-2.507 2.5742-3.6094-2.8494-.051-5.6268-.2287-8.4394-.2676z"></path><path id="locationMap-KE" d="m526 371.97c-.7978-.01-1.6023.049-2.4121.1894-3.0908-.1159-6.1537.3588-9.1972.8438-2.2079.4652-4.5011 2.2705-4.9395 4.0156-.3674 1.4624.7835 2.7895 2.2871 4.1582 2.3881 2.7056 6.4956 4.4463 6.6816 8.5547.8606 5.2529 1.1163 11.317-2.582 15.617-2.7297 3.9626-7.0105 8.685-9.1074 13.049 3.0525 6.4872 18.212 9.9608 24.404 15.973 3.4599 3.2126 9.1028 4.1728 11.312 8.5039.015 5.9919 7.5528 5.5695 11.24 8.4023 1.0835-1.0872 5.5384-2.374 5.3828-4.75-.7316-3.2287-.2135-7.0094 3.5782-7.6855-.019-3.3544 2.8904-4.4736 4.9707-6.4492-.3947-3.3212 2.8125-3.4823 5.1191-3.6153-10.376-12.366-6.2127-18.389-6.9277-34.045-.5879-4.4558 3.6265-6.4072 6.3574-8.875 1.3308-1.4499 3.1086-3.3785 4.1191-5.0703-6.8122-3.3761-14.62-.7893-20.91 2.5156-5.1274 2.5322-11.092.011-14.391-4.2715-4.1603-3.4718-9.4013-6.9877-14.986-7.0605z"></path><path id="locationMap-KM" d="m587.4 513.18c-.4307.7122.033 1.5947.6853 1.9836.4416.5282 1.5857.6376 1.5565-.2876-.033-.7843-.5384-1.4633-1.0341-2.0346-.479-.5808-1.124-.4003-1.2077.3386zm-5.8-1.4c-.3877.5844-.2564 1.3693.3337 1.7587.5908.4144 1.6292-.1874 1.3132-.9355-.2135-.5139-.6249-1.2436-1.2771-1.1643-.1735.038-.3137.1746-.3698.3411zm12.5 4.6c-.497.5038-.4888 1.3222-.033 1.8493.2331.6023 1.1714 1.343 1.6498.5778.2179-.6526-.045-1.344-.1875-1.9871-.074-.5553-.7216-1.3402-1.2224-.6952-.075.08-.1425.1667-.2071.2552z"></path><path id="locationMap-LR" d="m145.6 344.55c-.8734-.044-1.9867.2978-3.4824 1.2109-3.5071 3.7046-6.9043 8.0114-10.275 11.006 1.605 1.1273 4.0614.8845 5.5449 2.0859 1.2288 1.5207.8117 4.1475 3.0645 4.8204 1.1637 1.2194 2.3044 2.5083 4 3.0019 2.0022.7846 2.4564 3.1062 3.8789 4.5098 1.4971 1.4363 3.7045 1.8983 5.1211 3.4668 1.7789 1.9232 3.7916 3.6806 6.2402 4.6914 1.1314.472 3.8759.9965 4.5976.1992-.3713-1.9313-1.5345-3.7725-.8222-5.709.6241-2.2667 1.3517-4.8979-.053-7.0332-2.1138-3.3622-6.5013-4.7122-8.0253-8.5234-.7317-2.0161-.11-4.4064-.1465-6.5118-6.5618.4076-5.8578-7.0251-9.6426-7.2148z"></path><path id="locationMap-LS" d="m449.9 664.98c-3.0567 2.2709-6.9039 4.0411-7.3791 8.1809-2.6704 4.1546 2.8078 6.5053 6.1979 5.4266 3.5859-2.1605 8.4652-2.0599 11.249-5.231 1.9994-3.886-2.3149-7.7474-5.1601-9.849-1.7295-.8103-3.9148-.1671-4.9074 1.4725z"></path><path id="locationMap-LY" d="m318.9 128.09s-3.5264 2.3415-4.4786 4.2402c-1.2067 2.0442-1.9331 5.0589-4.5957 5.6191-2.8864.56765-4.4538 3.7918-3.8691 6.5293 1.0267 2.4804-1.5456 6.4737-2.8672 7.0684 1.0305 10.928.3531 23.194.8457 34.197-4.3829 6.7065 2.6173 10.532 4.1309 16.287 1.0628 5.1999 13.869 3.8493 12.76 10.438 5.984.5848 14.556 2.443 18.209 7.7696 2.3094 4.3027 6.6581-.249 9.4102.3457 1.0802-2.8479 8.4228-5.3362 11.514-4.0254 14.99 8.0678 30.626 14.899 45.898 22.42 4.7536 1.7925 9.0704 5.567 14.457 5.1426 3.2138.2848 6.6206.118 7.9414-.098-.2847-5.7578-.9119-13.488-1.209-19.264-1.802-30.904-1.0035-58.044-2.3379-85.771-3.8157-6.0262-12.889-.99434-16.92-4.8555 1.1188-7.5804-8.4202-5.1187-13.06-4.791-5.0737-.66759-12.852 1.5123-12.33 7.9004 1.4288 4.38 1.1162 9.4979-3.7539 11.299-3.0502 7.7529-9.754-.5497-13.875-2.252-4.6864-4.5189-13.405.32659-16.781-4.9805-.1806-6.7443-5.8881-10.768-12.254-10.316-5.1481-2.8842-11.476-1.6031-16.834-2.9023z"></path><path id="locationMap-MA" d="m179.5 103.8c-.7866.0912-1.5219.61513-2.1836 1.8555-.5225 3.8043-2.1181 6.6132-4.6758 9.4824-1.1773 3.7687-4.9102 6.1274-8.3379 7.5938-3.8052 1.0136-7.3477 2.6536-8.8496 6.4004-2.244 2.722-5.5469 4.9679-6.5273 8.5176-.5237 3.3938-2.843 6.841-1.8535 10.279 1.6871 3.6897 1.4579 8.065 1.332 11.953-3.4782 2.3204-6.1619 5.9352-10.228 7.4551-3.6488 2.2239-7.1052 5.008-11.014 6.6543-3.0868-.3644-8.4852 1.1462-9.0367 4.1879 13.085-.1656 26.492.1671 39.578.0.2347-6.2065.433-13.993 8.332-14.037 5.9953-5.3736 14.824-1.1604 20.494-6.7676 2.151-5.4703 9.3817-6.3754 11.273-11.096-.3761-12.397 14.664-1.9844 18.602-9.0996-1.103-6.9391-1.9878-13.707-4.8887-20.512-1.2126-3.4396-1.1286-5.8828-1.2226-9.3457-4.4681-.29091-9.5348-4.2291-13.389.0586-3.2437-1.3648-6.2886-.60301-9.584-.29297-2.6449.0665-5.4607-3.5628-7.8203-3.2891z"></path><path id="locationMap-MG" d="m641 530.68c3.1643 4.2097.3002 9.612 2.5938 14.198 1.5687 3.5153 1.0682 7.9572-3.4938 7.902-2.1234 3.9473-10.152 7.0758-6.3 12 .6359 4.2367-1.7588 8.7074-3.2266 12.858-1.3287 5.6106-5.5934 9.9298-6.6611 15.63-.9852 5.0785-4.2059 9.1656-6.0576 13.894-2.2358 4.4259-6.3512 7.8209-5.594 13.348-.9886 5.5282-3.2521 11.187-8.1107 14.357-4.4186.5018-8.9571 2.5725-13.55 2.7125-4.8979-.187-10.622-6.4305-7.15-10.988-3.2731-3.016-2.6415-7.3042-3.7568-11.167-.5897-4.1217-1.6666-8.3993-1.0932-12.546 3.1389-4.1205 5.8627-8.6501 10.067-11.82 3.8227-3.7785-1.0776-8.7175.6412-12.982.2909-4.5553-2.7053-8.4874-1.7834-13.185 1.6564-3.8851 5.9003-6.502 5.2422-11.091 2.5795-3.1761 7.1193-1.1739 10.45-3.5375 3.6706 1.4655 5.5637-2.6044 9.1828-1.8375.7379-.8197 1.3714-6.9994 2.4203-4.0328-.1897 3.9451 3.68.3889 1.8-1.9797.8856-2.813 3.1312-4.4857 4.3703-3.2719 4.9614-1.1598-.4407-8.8327 5.8219-8.4391 2.417-2.4299 5.5945-4.7806 3.1828-8.5922 2.3755-.8485 5.582-9.0313 6.5469-3.4109-2.8298 4.3714 5.6362 4.5393 3.1078 9.1422.2217 1.0358.7782 1.9616 1.35 2.8375z"></path><path id="locationMap-ML" d="m187 200.52c-4.627.958-11.321.9478-14.75 1.7774 3.4114 22.171 6.1881 56.678 8.6582 78.971.2965 2.8009-.083 2.8277-4.4082 3.0957-9.2438.023-18.566 1.3838-27.742-.09-5.801 2.1903-11.861.9563-17.885.4473-3.3707.8043-5.2532 2.2203-5.1036 4.4785 1.8488 1.3011.9151 4.151.5313 6.2012-.4779 3.437 1.8887 8.3414 4.5508 11.104 2.1717 1.2568 4.0753 5.0099 2.9687 7.2988 1.609.9728 2.8962 2.7981 4.8379 2.1719 2.0666-.7377 4.3331-.91 6.3008-1.9004 1.8121-1.3152 4.7123-4.1096 6.5312-1.2852.795 2.4979 1.5808 5.0529 3.3028 7.1055 1.872 2.1922 3.4547 5.4641 4.0195 8.2285 6.6367-2.9748 15.104.7229 21.432-1.0136-.5856-1.2096 1.6176-4.6339 2.2657-6.336 2.0199-1.8445 2.9153-4.2884 3.2539-6.9551 1.2927-2.5965 3.0309-4.9934 4.8261-7.2734 2.1915-1.9779 5.8779-.9321 7.8809-3.3457 1.3042-2.1334 3.0154-3.9524 5.1973-5.2227 2.4558-1.5234 2.5547-5.1909 5.8281-5.6523 3.1181-.3008 4.8719-3.3172 7.7988-3.9707 3.0602-.1648 9.4727-.502 9.4727-.502 2.4592-1.3395 8.8428-1.073 12.748-2.3457 5.5881-.4329 11.45-1.4094 17.193-2.1816 3.0158-4.7632 4.2308-10.399 4.8086-15.938.7799-5.7183-.5633-9.8983.4004-14.846.0.0-11.293 1.8951-9.3594-4.4668-3.101-6.9672-13.023-3.0323-16-9.6992-3.8927-5.5731-10.762-8.4012-15.9-12.801-10.832-7.989-22.69-17.257-33.658-25.055z"></path><path id="locationMap-MR" d="m157.6 180.78c-.067 1.649-.1039 2.7569-.1718 4.4062-.5712 2.8278.8225 6.8248-1.5215 8.7793-6.6804.346-13.394.028-20.084-.057-2.7273.2191-7.3275-1.1817-7.9981 2.6426-.2443 5.4584-.051 10.939-.1132 16.406.6724 3.7978-4.5612 3.1398-6.3457 5.3984-1.9645 2.7566-1.9486 6.4005-2.0196 9.6563-.2002 2.0667 1.0888 5.991-2.1386 6.1426-7.5911-.1541-15.178-.6329-22.766-.9336-3.0214-.3927-7.4555.8155-9.7246-.1114 1.4997 2.7187-.2894 7.977 4.2012 5.7696 4.9578 1.9737 2.5041 6.7483 1.0215 10.184 3.1045 4.0814 3.7202 9.4755 4.6777 14.416-1.5049 3.5484-3.5015 9.208-2.5371 13.088 1.883-1.4869 4.4687-.652 6.6074-1.5879 2.0161-1.3032 4.7748-1.6747 6.9961-.7246 1.7249 1.3202 2.7207 4.2034 5.4121 3.7344 2.0606-.255 4.5971.3412 5.416 2.4765.8109 1.9937.3555 4.899 2.7696 5.8945 1.9615.8401 4.6907 1.8179 6.5429 2.8438-.1496-2.2582 1.7329-3.6742 5.1036-4.4785 6.024.509 12.084 1.743 17.885-.4473 9.1767 1.4734 18.498.1129 27.742.09 4.3251-.268 4.7047-.2948 4.4082-3.0957-2.4701-22.292-5.2468-56.8-8.6582-78.971 3.4288-.8296 10.123-.8194 14.75-1.7774-9.6474-6.892-20.331-12.727-29.453-19.744z"></path><path id="locationMap-MU" d="m691.7 596.98c.896-.2631 1.7607-.7891 2.7226-.7125.6834.2023.9493 1.0482.8788 1.6934-.046.6566-.6797.8942-1.2091 1.0715-1.5434.6798-2.4783 2.3096-4.1439 2.7695-.4742.2918-1.0824-.1679-.867-.7071.1392-1.4375.7974-2.8977 1.9755-3.7733.1996-.1391.4153-.2557.6431-.3415z"></path><path id="locationMap-MW" d="m504.9 495.58c-1.6277.02-3.3565.2136-4.8867.084 2.0959 2.2844.5163 7.0255 2.2051 9.2871 1.9121 2.0766.9263 5.0186 1.0781 7.5078.5188 2.9631 1.5342 6.336-.1426 9.1094-1.7783 1.5166-4.3176 2.7214-3.3574 5.6035.8246 2.3018-1.0739 4.5131-1.1241 6.8043 2.1187 1.0489 6.6414.445 8.2667 2.3793 1.2681 1.6681 1.4092 4.109 3.4219 5.1465 1.3788 1.8813 2.0886 4.6232.8359 6.7695-1.2934 2.1318-2.5758 4.7611-.7695 7.0391 1.137 1.6667 3.5341 5.1465 5.4902 2.4609.5293-2.0971 1.7893-3.9389 3.6797-5.0996 2.6216-2.2882 3.3976-6.2913 2.4023-9.5391-.6983-2.234-3.3843-2.8932-4.2011-5.08-1.1357-2.4176-1.6565-5.1808-3.9258-6.9043-2.5105-2.2224-1.8231-5.8458-2.168-8.8301.026-2.3453-.6633-4.5712-1.1758-6.8223-.042-1.8768 1.989-2.4148 2.9278-3.9179-2.5374-3.8571-1.8284-7.8508-2.6406-12.348-.7704-3.2336-3.2032-3.6835-5.9161-3.6504z"></path><path id="locationMap-MZ" d="m560.9 506.48c-4.8026 1.6269-12.949 2.7821-17.676 4.7617-4.9275 1.8975-10.543 2.7317-15.699 1.5235-3.8891-2.9709-10.028-.8799-14.041-1.2168-.9388 1.5031-2.9698 2.0411-2.9278 3.9179.5125 2.2511 1.2018 4.477 1.1758 6.8223.3449 2.9843-.3425 6.6077 2.168 8.8301 2.2693 1.7235 2.7901 4.4867 3.9258 6.9043.8168 2.1868 3.5028 2.846 4.2011 5.08.9953 3.2478.2193 7.2509-2.4023 9.5391-1.8904 1.1607-3.1504 3.0025-3.6797 5.0996-1.9561 2.6856-4.3532-.7942-5.4902-2.4609-1.8063-2.278-.5239-4.9073.7695-7.0391 1.2527-2.1463.5429-4.8882-.8359-6.7695-2.0127-1.0375-2.1538-3.4784-3.4219-5.1465-.8127-.9671-2.3505-1.2996-3.9492-1.5215-1.1985-.1663-2.4313-.2706-3.4199-.5332-.3296-.088-.6316-.1931-.8965-.3242-2.6154.6746-5.0191 2.3005-7.7012 2.8457-4.4519.8574-8.6103 2.6087-12.562 4.7871-1.9831.8322-2.9697 2.8139-4.5605 3.834-6e-4.0.0.0.0.0.3531 1.2026-.354 1.5931-.1309 3.2441.5678 3.1428 4.1579 2.9317 6.5938 2.9317 3.2726-.1456 5.1717 2.7004 7.7519 4.1172 2.4018 1.9015 6.2118.5774 8.1133 3.0566 1.4985 3.1287 1.9111 6.7313.9903 10.098-.3472 4.6349-1.7639 9.3037-.6934 13.943 1.1508 3.0898-1.4465 5.4571-3.5625 7.2324-2.6691 2.5615-1.7287 6.7044-3.7187 9.5801-3.0483 2.2063-4.6709 6.2887-5.3633 10.014v.01c1.4863 6.4441 4.7866 16.827.6582 22.723-.3937.8729-.3627 1.9746-.7784 4.3807 1.6433 1.816 2.5334 4.6427 3.2084 5.9827.7338.4361 3.0595-.6667 3.8426-.6092 2.6127-3.2008 5.8388-6.8795 9.368-8.7523 4.4198-1.2724 9.8338-3.885 11.547-8.7871 3.5794-5.5626 1.9603-12.478 2.4414-18.854-1.0033-5.5756-1.0898-11.499-3.9082-16.539 3.2817-3.7237 8.1053-7.5932 10.482-11.844-1.6182-5.3859 8.2319.716 7.459-5.7363 3.0718-3.4992 6.4922-6.825 9.7402-10.064 6.1853.014 12.104-3.8401 16.774-7.4062 2.4723-3.546 10.591-7.5443 6.3925-12.16-3.0423-5.2989 1.9781-10.225.834-15.896.539-5.9894-1.0659-13.072-1.0117-19.6z"></path><path id="locationMap-NA" d="m332.4 559.78c-3.8252-.1197-7.1906 1.589-9.9961 2.1992.8071 5.1322-.8082 8.8871 2.4023 12.9 4.1529 3.3933 7.3833 7.8849 8.1035 13.242.2526 6.1169 6.8268 7.9675 7.7403 13.904 3.6209 4.0532.3433 9.0582 1.2109 13.631 1.0474 5.008 1.5133 10.149 3.2461 15.022 3.4002 4.9766 1.5735 11.109 3.1836 16.467-1.2114 5.5654 2.5871 10.723 6.3672 14.383 4.9168-10.321 9.6544 1.7843 15.809 2.8692 4.9554.2867 11.601 1.2682 13.84-4.5176 1.127-10.842.4118-22.455.6094-33.352.1818-7.0614-.021-14.378.6055-21.422-1.0624-5.1416 2.7555-3.6303 6.9863-5.0273s1.7076-9.3802 2.6582-13.871c.9345-5.1918-.8891-11.666 3.207-15.674 4.9412-1.7677 18.349-1.3754 23.553-2.084-.2357 3.6011 11.104.8339 10.582-2.7383-4.128-.2684-9.4421-.3757-13.408.6309-7.8653 1.261-21.274 1.8846-27.592.029-5.1308-.5896-10.44-.8584-15.117-3.291-8.4194-2.1173-17.199-.5941-25.772-.6641-5.2687.1499-10.537-.8478-15.5-2.2246-.9305-.2582-1.836-.3845-2.7187-.4121z" class="highlighted"></path><path id="locationMap-NE" d="m320.9 212.47c-11.968 11.277-27.712 23.792-40.807 32.873-4.5055 3.74-12.026 7.188-18.104 7.2031-.9637 4.9474.3795 9.1274-.4004 14.846-.5778 5.5388-1.7928 11.174-4.8086 15.938-5.7436.7722-11.605 1.7487-17.193 2.1816-3.9052 1.2727-10.289 1.0062-12.748 2.3457-9e-4.0.0.0.0.0-6e-4.01-.1921 1.7632.068 4.0958.2609 2.3364 2.7439 6.4017 5.2793 8.0605s1.3694 5.111 4.168 6.3848 6.3666 4.3888 7.9003 5.7129 2.5738.1916 2.8438.6836c2.0068-1.1558 6.1839-1.1512 8.2539-1.6876 3.1326-1.8541 4.866-3.391 5.6054-6.83 2.2315-5.5816 8.7794-3.6647 13.426-4.2617 5.2838-.5822 7.3671 6.9992 13.027 4.0898 5.5047-1.9473 9.1258 3.9727 14.207 4.4043 5.2669-.853 10.143-6.7701 15.896-3.5898 5.6573 1.4304 12.854-4.1714 18.764-6.7657.3047-3.6421 2.5968-9.0599 3.377-13.391 4.4554-4.7815 10.793-7.7357 13.975-13.688 2.1006-4.7062.5582-10.078 1.1973-15.053.3764-5.4826 4.9044-11.194 1.7031-16.572-7.3083-2.5755-1.7188-11.434-8.0117-18.869-2.7521-.5947-7.1008 3.957-9.4102-.3457-3.6526-5.3266-12.225-7.1848-18.209-7.7696z"></path><path id="locationMap-NG" d="m336.2 298.16c-5.9101 2.5943-13.106 8.1964-18.764 6.7657-5.7532-3.18-10.63 2.7368-15.896 3.5898-5.0813-.4316-8.7027-6.3516-14.207-4.4043-5.6602 2.9094-7.7435-4.672-13.027-4.0898-4.6464.597-11.194-1.3199-13.426 4.2617-.7394 3.439-2.4728 4.9756-5.6054 6.83-.3224 4.2593 3.5453 9.7855 2.1425 14.15-1.2107 5.2101-3.7886 10.08-6.9316 14.332-2.1654 4.6194-1.3575 9.9023-1.6777 14.844.4217 3.3326-.579 5.0757-.3594 7.4102 4.9525-2.1472 13.722.5333 17.369 3.7344 2.195 2.5344 2.9974 4.7953 6.3282 5.2812 1.5643 3.0012 1.7177 6.6638 6.1503 6.6914 4.4776 1.5642 12.914-2.7074 17.418-1.4531.8987-2.4948 3.3505-4.0545 3.8164-6.5547.047-4.7655 1.999-9.9373 6.4336-12.254 4.3209-2.7056 8.8018 2.1323 13.213.6621 4.5428-3.1441 5.9274-8.8603 8.4102-13.502 2.4964-3.529 4.8676-7.0686 5.9648-11.273 1.3279-3.153 5.4895-5.3641 4.0312-9.2812-.2961-4.3872 5.3386-5.5315 6.1446-9.3809.043-3.897-4.1459-9.1453-3.0781-12.15-1.9991-1.0228-2.6973-4.2032-4.4493-4.209z"></path><path id="locationMap-RE" d="m672.2 603.98c-.2209 1.6752.5816 3.6433 2.2482 4.2514 1.1023.3046 1.4296-1.0065 2.2139-1.3927.9823-.132 1.7356-1.3438.8977-2.1193-.6322-.5802-1.4854-.8214-2.2143-1.2585-.8038-.3715-1.8059-.7345-2.651-.3024-.2723.1857-.4615.4925-.4945.8215z"></path><path id="locationMap-RW" d="m483 423.65s-5.195.2976-7.2402 1.4961c-1.1718.6939-2.1262 1.8017-3.5019 2.1055-.8676.2368-1.6477-.1614-2.5235-.334-.9103 2.9795-.8107 5.9692-.4101 9.0957 3.1357.3608 1.8715 1.2158 3.5839 1.8605 1.7124.6446 3.3498-1.1146 4.6817-2.2648.4253-.5606 1.7799.04 2.0957-.4902.2951-3.4633 4.3083-7.4068 3.3144-11.469z"></path><path id="locationMap-SC" d="m659.3 476.68c.037 2.047 2.1099-.5345.1006-.1225zm4.3.2c-.2146 1.7707 2.9704-.266.9666-.6439-.3811.014-.8561.2568-.9666.6439zm26.4-8.8c-1.2269 1.7254 2.9503 1.7763 1.8973.2505-.3765-.5767-1.701-1.4653-1.8973-.2505zm.4-7.1c-1.9796.9996 1.0277 1.988 1.8141 1.1734-.042-.7331-1.0486-1.9901-1.8141-1.1734zm-10.8-7c1.0032-.7999 2.6834.682 1.0814 1.1742-.6844.415-2.0401-.6118-1.0814-1.1742zm10.012-.038c-.2508-1.0945-2.1089-1.7572-3.203-1.459.5068 8.8962 5.5531 6.7071 3.203 1.459z"></path><path id="locationMap-SD" d="m514.7 213.03c-1.2016 1.1115-6.8114 1.033-7.7832 3.5039-4.0383 1.5795-3.0507 9.297-6.8047 9.334-3.833-3.719-9.3832-3.1222-14.266-3.2012-19.223.6034-39.559 1.7163-58.791 2.0918.2971 5.7756.9243 13.506 1.209 19.264-1.3208.2154-4.7276.3816-7.9414.098.5214 10.502 2.7459 25.78 2.3984 36.314-18.303 9.9066-5.8342 32.072-7.75 40.93 3.6439 1.8313 10.596 6.3759 8.0938 11.113-3.1631 3.4733 1.312 4.4803 3.9961 6.5449 1.8175-.2 3.0315-2.7647 2.7031-5.9453-.8844-6.4388 5.9699-12.944 11.559-7.8574 1.1396 4.06.791 11.303 7.2168 7.9707 5.9214-.3949 12.92 2.5598 18.022-1.7949 3.631-5.6001 10.148-4.3949 14.029.068 6.7246 2.2167 10.143-4.7743 11.174-10.186.9943-3.7071 2.8063-7.2865 3.0996-11 9.0034-2.093 2.2744 12.546 11.051 12.525 2.245.3072 3.2811 5.7151 3.9609 5.9257-.4199-2.5999 1.7203-5.8644 5.5157-8.3339s2.0802-10.037 6.1113-13.914 8.2912-8.5733 5.8203-14.275c-.8057-1.6552-.6087-2.4388-.7617-3.0781.5898-5.8781 3.2921-9.6428 3.2031-15.551.1515-2.6329.1087-5.9783 3.4004-6.5508 2.9998-2.1812 4.4317-5.7605 6.7813-8.5566-2.4642-2.1039-2.6419-4.8847-6.1563-5.5293-4.167-1.6372-.8932-6.9578-4.3516-9.8438-.7983-6.1172-2.3843-11.747-2.7226-18.066-4.3582-3.1412-7.883-10.504-12.016-12z"></path><path id="locationMap-SH" d="m122.6 488.15c-.067-.01-.1345.0-.1972.025-.1662.1163-.2724.3033-.336.4922-.043.1396-.072.2959-.014.4356.021.061.078.098.1367.1171.1647.057.3396.0.4961-.057.1853-.07.3652-.1537.5586-.1992.072-.02.1244-.091.1152-.166-.015-.1423-.1113-.2599-.207-.3594-.1501-.1469-.3399-.267-.5528-.2891zm53.039 63.68c-.083.0-.1666.04-.2285.1-.083.069-.1504.1555-.209.2461-.3425.4517-.7542.8584-1.0176 1.3653-.1488.2814-.1956.6333-.045.9238.1929.3458.441.6713.7617.9082.238.1782.5712.2707.8477.125.3409-.1779.5682-.531.6445-.9024.131-.5984.083-1.2259-.066-1.8164-.09-.3145-.1986-.6519-.4551-.8711-.066-.058-.149-.083-.2324-.078z"></path><path id="locationMap-SL" d="m130.5 331.7c-1.2463-.04-2.496.448-3.6211.9629-1.5821.4379-3.2232.8482-4.6172 1.7559-2.2242 2.0903-4.5145 4.0483-7.0352 5.7871.8294 1.1531 1.6849 2.6542 2.2774 3.9433 1.0509.8129 2.0489 1.7308 2.3164 3.1289.591 1.5269 1.5367 2.8879 2.1601 4.3946.3153 1.1369.8664 2.5776 2.2442 2.6562 1.8013.335 3.6657.236 5.4219-.291 1.2735-.57 2.6681 1.6905 2.1406 2.7246 3.3711-2.9945 6.7683-7.3013 10.275-11.006-1.2609-.7486-2.9579-.4545-3.6602-1.082-1.3317-.7557-1.6845-2.4337-1.2969-3.8242.3156-1.3335.4435-2.7904-.2636-4.0293-.5229-1.0349-1.1435-2.0232-1.7813-2.9844-1.0202-.7503-2.2476-1.2182-3.3203-1.9023-.4094-.1495-.8248-.2209-1.2402-.2344z"></path><path id="locationMap-SN" d="m103 273.72c-1.5274-.01-3.0778.4414-4.3379 1.2559-2.1387.9359-4.7244.101-6.6074 1.5879-.2431 1.4911.055 3.9571-1.5254 4.8496-.6797 2.2025-2.8923 3.5383-4.5586 4.9922-.9841 1.0044-4.6776 1.5604-2.3867 3.3008 2.6763.7427 1.5363 4.0513 3.5664 5.455 1.1671.5191 1.9861 2.0059 1.6875 3.2637 2.2486-.3806 4.2626-.6204 6.4902-.2539 3.8071-.154 7.8092-1.4021 11.504.098 1.607.9767 3.8251 1.4791 4.5137 3.414.1359 2.6914-3.1302 3.2481-5.2148 3.0313-2.729-.095-5.4637-1.0642-8.17-.2188-2.3587.6545-4.8 1.1486-7.248.7051-.815.021-2.2803-1.071-2.7559-.7656.4196 1.3964 3.5074 4.554 3.8985 6.7265 3.8052-1.5003 6.9688-3.5874 10.713-3.623 3.7971-.1126 7.4683.1823 11.268.25 3.366.043 5.9823 2.7221 8.3262 4.5488 2.8334 1.8534 7.6608-.2671 11.688 1.4668 1.1066-2.2889-.797-6.042-2.9687-7.2988-2.6621-2.7621-5.0287-7.6665-4.5508-11.104.3838-2.0502 1.3175-4.9001-.5313-6.2012-1.8522-1.0259-4.5814-2.0038-6.5429-2.8438-2.4141-.9957-1.9587-3.9008-2.7696-5.8945-.8189-2.1353-3.3554-2.7315-5.416-2.4765-2.6914.469-3.6872-2.4142-5.4121-3.7344-.833-.3563-1.7417-.5259-2.6582-.5313z"></path><path id="locationMap-SO" d="m646.6 309.16c-.1787-.014-.3643-.014-.5567.0-.7102.064-1.5119.3485-2.3945.9121-4.0815 3.2798-9.5997 2.2839-14.199 5-5.2841.9001-11.028-.8536-14.977 4.0371-4.8228.515-9.8763.8051-14.012 3.5879-3.3963 3.5461-8.2406 3.255-11.412-.625-1.5671-.712-3.542-4.5335-4.8379-5.502-.8323.5813-1.6006 1.2775-2.0586 2.2012-1.0552 3.6371 6.7059 10.396 7.8965 13.301 3.7811 2.6862 8.2407 4.4664 12.547 6.1992 4.6663 1.7862 9.1711 4.9897 14.275 4.8516 3.5549-1.6154 8.5232 1.0645 3.33 3.7773-7.2406 7.629-14.636 15.161-21.211 23.395-3.563 3.6381-10.016.3143-12.941 4.7187-1.8736 3.7736-6.7568 3.6766-9.752 5.7676-1.0105 1.6918-2.7883 3.6204-4.1191 5.0703-2.7309 2.4678-6.9453 4.4192-6.3574 8.875.715 15.656-3.4485 21.678 6.9277 34.045 3.2452-4.5145 7.4689-8.7086 9.8731-13.682 4.562-2.5166 7.6282-6.7631 11.695-9.9238 4.0701-2.7489 6.8104-6.7332 10.832-9.504 4.6268-3.3488 7.9995-7.7911 11.84-11.894 4.0115-4.05 6.9164-8.7788 10.475-13.195 4.0623-3.8104 5.3379-9.372 9.0566-13.473 2.4709-4.2527 2.1573-9.1191 3.0293-13.627 4.1197-3.1683 5.0709-8.3416 6.6992-13 1.0845-5.0533 2.7572-9.3398 2.9629-14.4 1.187-3.1124.069-6.7075-2.6113-6.916z"></path><path id="locationMap-SS" d="m496.4 310.08c-.4474.01-.949.072-1.5117.2032-.2933 3.7135-2.1053 7.2929-3.0996 11-1.0306 5.4112-4.4492 12.402-11.174 10.186-3.8809-4.4635-10.398-5.6693-14.029-.068-5.101 4.3547-12.1 1.4-18.022 1.7949-6.4258 3.3322-6.0772-3.9107-7.2168-7.9707-5.5887-5.0871-12.443 1.4186-11.559 7.8574.3284 3.1806-.8856 5.7453-2.7031 5.9453 3.9005 3.0436 6.2874 8.0491 11.604 10.654 4.7258 1.1416 4.7528 6.6475 8.1211 9.2305 3.8439 2.0849 5.5417 7.0344 7.9551 10.301 3.2417 11.695 18.872 4.0109 27.48 12.146 3.3355-.027 2.8006 1.2165 7.6133-.8203 5.2685-1.8416 11.211.9014 16.25-1.7715.972-.5986 2.5157-.867 3.293-1.7441.4384-1.7451 2.7316-3.5504 4.9395-4.0156 3.0435-.485 6.1064-.9597 9.1972-.8438-.3713-4.5288-4.5468-7.9388-7.1269-11.24-.9136-3.8776-3.5347-6.8629-6.3223-9.5918-2.6018-3.3618-10.471-1.5053-9.3438-7.4922 1.8699-3.9587 8.555-2.0759 9.2813-7.1582 2.1631-4.5761.9796-3.9362-.127-7.9473-.6798-.2106-1.7159-5.6185-3.9609-5.9257-8.2279.019-2.8276-12.845-9.5391-12.729z"></path><path id="locationMap-ST" d="m278.9 412.28c-.5094.5646-.5799 1.602.088 2.0814.4279.5137 1.1667.6165 1.7064.2283.6308-.1893 1.441.1332 1.9063-.489.6128-.4664-.046-1.0903-.4697-1.386-.7465-.597-1.7729-.9647-2.7264-.7456-.1914.057-.3726.159-.5045.3109zm7-10.6c-.4795.4453-.7674 1.34-.1873 1.8238.6484.5362 1.8819.2148 1.9792-.6939.01-.635-.4876-1.3859-1.1819-1.3662-.2218.013-.4302.1103-.61.2363z"></path><path id="locationMap-SZ" d="m480.6 636.58c-1.6695-.081-3.2422.235-4.7246 2.7559-3.103 4.474-.3711 12.305 5.1035 13.143 4.4011.2646 5.5712-6.468 5.9414-9.7344-.675-1.34-1.5657-4.1664-3.209-5.9824-1.0732.057-2.1096-.1334-3.1113-.1817z"></path><path id="locationMap-TD" d="m357.9 216.2c-3.506-.01-8.5169 2.0708-9.3945 4.3848 6.2929 7.4356.7034 16.294 8.0117 18.869 3.2013 5.3783-1.3267 11.09-1.7031 16.572-.6391 4.9748.9033 10.346-1.1973 15.053-3.1816 5.9518-9.5192 8.906-13.975 13.688-.7802 4.3307-3.0723 9.7485-3.377 13.391 1.752.01 2.4502 3.1862 4.4493 4.209 3.3847-.1561 4.4169 6.7004 8.1484 6.9824 5.3666 1.9182-1.5273 5.4619.5508 8.8282 1.4645 3.1559 4.5147 9.6824-1.1836 10.215-3.5187-.2331-10.394 2.7192-6.377 6.7656 7.1121 4.8983 11.792 5.9529 15.051 14.861 4.5327-.699 12.769-.056 16.838-2.502 3.6356-1.0715 8.5104-1.8712 8.8867-6.5508 2.9975-3.8653 9.0856-1.7388 12.988-4.4375 3.6413-2.6287 6.4214-6.1918 8.668-10.07 1.9589-3.445 7.0329-6.3984 10.738-5.0918 2e-4-6e-4-1e-4.0.0.0 1.9157-8.8576-10.553-31.021 7.75-40.928.3476-10.535-1.877-25.812-2.3984-36.314-5.3866.4244-9.7034-3.3501-14.457-5.1426-15.273-7.5209-30.908-14.352-45.898-22.42-.5795-.2458-1.31-.3582-2.1191-.3594z"></path><path id="locationMap-TG" d="m233.4 321.93c-1.1975.3734-2.3292.407-3.707.334-.1905 1.5525-.8466 3.4347-.6289 5 .5958 2.1241.156 4.3152.3809 6.4727.041 2.4352 1.7286 4.6877.957 7.1406-1.0328 3.161.9968 6.2178.6484 9.3828-.7828 2.3357-1.1794 4.8074-1.0273 7.2793-.092 2.7577 2.1361 5.5564 2.3203 8.2773 1.5881-.6568 5.3645-2.2359 7.1445-2.291.0-1.0458.8492-5.2372.9785-7.873-.4919-2.5319-1.4748-4.9894-1.3398-7.6133.065-3.2844.061-6.5711.1055-9.8555.3824-4.1936-1.5365-8.0419-2.5371-12.004-.1965-1.7807-1.7792-3.2937-3.295-4.25z"></path><path id="locationMap-TN" d="m318.9 128.09s-4.1822-3.556-6.2485-4.7775c-1.7355-2.088-4.7016-1.7732-6.6828-3.4475-1.9734-2.5883 1.4192-5.8597 3.7793-6.8289 2.0209-1.4542 3.6327-4.7358 2.1082-7.0586-1.5889-2.1504-5.2618-2.0037-6.1203-4.7766-.6421-2.4033 1.1326-4.3113 1.9828-6.3516 1.1226-1.1893 3.4631-4.7944.1629-4.3762-1.3201 1.6826-4.6961 3.3417-5.0713-.0213-1.1311-2.4064-4.1781-3.4066-6.4271-1.8758-1.8348 1.7652-8.6358.22044-10.193 2.6091 1.0931 2.4429-.3095 3.0234.8062 5.8085.2818 3.55 3.8361 5.9076 3.8132 9.5494 1.0666 2.1408 2.8353 5.0021.6684 7.1434-1.4114 1.7182-4.4603 2.2619-4.6309 4.8445.9375 2.5008-.218 5.2929.8717 7.698 1.5666 1.9731 4.1783 3.1593 4.8736 5.7766 1.1041 1.9426 3.1077 3.1871 4.2755 5.1106 1.4598 1.8168 2.1402 4.0794 2.423 6.355 1.2452 2.2831 2.7358 5.699 3.7985 8.0741 1.3216-.59465 3.8938-4.5875 2.8671-7.0679-.5847-2.7376.9839-5.962 3.8703-6.5297 2.6626-.5602 3.3886-3.5746 4.5953-5.6188.9522-1.8987 4.4782-4.239 4.4782-4.239z"></path><path id="locationMap-TZ" d="m506.7 418.4c-5.4042-.061-10.1 4.505-14.631 6.877-2.4688 1.3212-7.4733-2.4698-9.0586-1.6309.9939 4.062-3.0193 8.0055-3.3144 11.469 4.9004 7.6535 1.6332 9.053-4.4063 15.08-1.9477 1.6315-6.4535 1.6289-7.0898 3.2149-.2577.6423.3124 1.707 1.6445 2.9961 2.1363 3.3149 2.1849 7.4717 4.1133 10.924 2.1991 3.8192 3.429 7.9943 4.9942 12.059.9002 1.6075 1.4004 3.7394 2.4785 5.2461 3.0321 2.9813 7.7127 8.192 11.367 9.7187s2.705.2179 7.2265 1.3184c4.0804.3457 9.5702-1.6074 10.803 3.5664.8122 4.4969.1032 8.4906 2.6406 12.348 4.0125.3369 10.152-1.7541 14.041 1.2168 5.1564 1.2082 10.772.374 15.699-1.5235 4.7264-1.9796 12.873-3.1348 17.676-4.7617-3.0318-6.0781-3.5113-5.5241-4.8749-8.3633-.734-4.8051-3.6169-8.9861-3.8945-13.852 1.0707-4.6073.8645-9.3262.6875-13.982 2.0821-3.9665-3.9809-6.4596-1.793-10.84-1.0047-3.0714 2.8262-6.5497 2.6484-8.2012-3.6874-2.8328-11.225-2.4104-11.24-8.4023-2.2097-4.3311-7.8526-5.2913-11.312-8.5039-6.1921-6.0119-21.352-9.4855-24.404-15.973zm49.551 39.994c-.178-.02-.3949.057-.6543.2539-3.2332 1.5011 2.1024 11.416 1.0859 5.4668.2211-.748.8141-5.5817-.4316-5.7207z"></path><path id="locationMap-UG" d="m509.4 377.02c-.7773.8771-2.321 1.1455-3.293 1.7441-5.0388 2.6729-10.982-.07-16.25 1.7715-4.8127 2.0368-4.2776.7933-7.6133.8203.7294 2.3713 4.5293 4.1267 3.8672 7.0586-1.1644 2.3711 1.8193 4.458.377 6.7637-.9209 2.8267-4.0426 3.7431-5.9629 5.7344-1.9232 1.5247-2.8132 3.727-3.4141 6.0098-1.2325 2.0597-2.9545 3.8684-3.4394 6.3124-.9332 2.6993.1828 5.4748.037 8.1954-.3756 1.8627-3.7354 3.7289-3.9844 5.4824.8758.1726 1.6559.5708 2.5235.334 1.3757-.3038 2.3301-1.4116 3.5019-2.1055 2.0452-1.1985 7.2402-1.4961 7.2402-1.4961 1.5853-.8389 6.5898 2.9521 9.0586 1.6309 4.5308-2.372 9.2267-6.938 14.631-6.877 2.0969-4.3638 6.3777-9.0862 9.1074-13.049 3.6983-4.3004 3.4428-10.364 2.582-15.617-.1858-4.1084-4.2935-5.8491-6.6816-8.5547-1.5036-1.3687-2.6545-2.6958-2.2871-4.1582z"></path><path id="locationMap-YT" d="m599.8 521.18c-.3877.5844-.2564 1.3693.3337 1.7587.5908.4144 1.6292-.1874 1.3132-.9355-.2135-.5139-.6249-1.2436-1.2771-1.1643-.1735.038-.3137.1746-.3698.3411z"></path><path id="locationMap-ZA" d="m463.8 602.88c-1.0068 4.5206-6.5049 4.3882-9.3574 6.5625-3.719 3.752-8.0091 7.1995-10.488 11.951-1.1011 3.2353-3.5758 3.869-6.5273 3.6054-3.1223 2.2122-1.2271 7.0639-3.9239 9.7383-2.3415 3.2805-7.003 1.4291-10.422 2.1387-4.3604 1.3312-6.4089-4.2169-10.113-3.5059-3.9665 3.2639-5.9674 8.3137-9.7442 11.803-2.174 3.4282-6.9326 2.345-10.006 1.0703-.5891-4.2495-.3967-8.6294-.5371-12.932.7525-4.2301-4.0979-6.4166-7.7676-6.7696-.1976 10.896.5176 22.509-.6094 33.352-2.2386 5.7858-8.8844 4.8043-13.84 4.5176-6.1542-1.0849-10.892-13.19-15.809-2.8692 3.6452 1.4444 2.0976 7.6825 4.6367 10.764.448 6.8948 5.1953 11.352 9.2129 16.375 6.0231 5.768-6.2736 11.73-2 16.912 2.3954 3.2071 4.7992 5.6029 3.9004 9.5996 2.0424 2.4909 6.6831.8072 6.6484 5.5762 4.3279 1.418 9.7086 4.9912 12.348-.5528 5.1812-1.1202 9.1299-6.2548 15.203-4.2226 5.4747-2.5943 11.654 1.9948 16.449-2.1231 5.09 2.6434 9.4457-4.7241 14.951-.9765 5.6591-2.7922 10.061-7.2973 14.088-12.014 5.5908-4.0039 8.3228-10.653 14.586-13.807 6.0975-2.7952 7.6264-9.5655 11.826-14.281 1.4471-7.2387 11.765-8.2059 12.607-15.883.6913-4.3528 2.3737-10.849 1.6563-14.742-.4269-.031-1.3047.2692-2.1407.4903-.6974.1843-1.3693.3175-1.7031.1191-.3702 3.2664-1.5403 9.999-5.9414 9.7344-5.4746-.8374-8.2065-8.6686-5.1035-13.143 2.3718-4.0335 4.974-2.4213 7.8359-2.5742.4157-2.4061.3856-3.508.7793-4.3809 4.1276-5.8946.8305-16.274-.6562-22.719.0-9e-4.0.0-.01.0-5.7686-2.122-12.864-4.5331-20.033-6.8125zm-10.346 60.299c.4636.028.9251.1393 1.3574.3418 2.8452 2.1016 7.1596 5.9636 5.1602 9.8496-2.7835 3.1711-7.6622 3.07-11.248 5.2305-3.3901 1.0787-8.8676-1.2731-6.1972-5.4277.4752-4.1398 4.3222-5.9088 7.3789-8.1797.7444-1.2297 2.158-1.8986 3.5488-1.8145z" class="highlighted"></path><path id="locationMap-ZM" d="m481.4 484.68c-.6869.4292-5.8617 3.8854-9.5235 2.5273-4.3746-1.4434-8.5449 1.5311-9.7148 5.7442-.5017 3.4314.5715 6.9563-.1699 10.312-2.9341 3.24-5.8583 8.0985-1.1231 11.469 2.0812 4.3144 7.0597 4.5163 10.709 2.9922 3.0976 2.3138 2.7336 9.754-1.5195 10.578-5.1385 1.011-8.4215-3.1974-11.674-6.3457-3.7417-3.4058-8.5706-5.9179-13.742-4.5722-7.1514.4583-12.777-11.04-19.299-9.7871-2.0746 6.1595.3691 16.97-3.3652 17.928-4.7165.055-9.441-.1261-14.15-.025-1.8637 3.2787-.4808 8.0794-1.0352 11.953.024 5.1327-.4202 10.325.062 15.418 4.1613 1.7022 9.2734 10.768 12.215 13.518 3.9661-1.0066 9.2802-.8993 13.408-.6309 3.0725-.2231 11.76 1.7238 14.344-.1601 2.7868-2.931 7.0649-3.1963 10.461-5.0743 1.9729-1.607 5.5017-3.3286 4.5097-6.4492-2.0267-3.3139 1.6684-5.4831 4.4747-6.164 1.31-.3214 5.3252-1.8287 7.5605-2.4141 1.5908-1.0201 2.5774-3.0018 4.5605-3.834 3.9522-2.1784 8.1106-3.9297 12.562-4.7871 2.6821-.5452 5.0858-2.1711 7.7012-2.8457.05-2.2912 1.9496-4.5029 1.125-6.8047-.9602-2.8821 1.5791-4.0869 3.3574-5.6035 1.6768-2.7734.6614-6.1463.1426-9.1094-.1518-2.4892.834-5.4312-1.0781-7.5078-1.6888-2.2616-.1092-7.0027-2.2051-9.2871-4.5215-1.1005-3.572.2083-7.2265-1.3184s-8.3351-6.7374-11.367-9.7187z" class="highlighted"></path><path id="locationMap-ZW" d="m473.8 545.48c-2.2353.5854-6.2499 2.0925-7.5599 2.4139-2.8063.6809-6.5014 2.8501-4.4747 6.164.992 3.1206-2.5368 4.8422-4.5097 6.4492-3.3961 1.878-7.6742 2.1433-10.461 5.0743-2.5839 1.8839-11.271-.063-14.344.1601 7.2866 6.4811 5.2685 14.189 10.094 17.451s4.5031 6.4543 7.6602 8.8965c2.1974.5276 5.1677 10.932 13.541 10.807 7.1696 2.2794 14.264 4.6911 20.032 6.8131.6916-3.7271 2.3157-7.8143 5.3661-10.022 1.9901-2.8757 1.0496-7.0186 3.7187-9.5801 2.116-1.7753 4.7133-4.1426 3.5625-7.2324-1.0705-4.6396.3462-9.3084.6934-13.943.9208-3.3664.5082-6.969-.9903-10.098-1.9015-2.4792-5.7115-1.1551-8.1133-3.0566-2.5802-1.4168-4.4793-4.2628-7.7519-4.1172-2.4359.0-6.026.2111-6.5938-2.9317-.2234-1.6529.4862-2.042.1303-3.2478z" class="highlighted"></path><path id="locationMap-AD" d="m238.9 36.52s-.6067-.61335-1.0849-.80639c-.547-.18149-.9692-.52635-1.492-.76995-.6266-.23621-2.152.15468-2.8202.0663.1671.64644.3224 1.3894.6776 1.9548.8278.7171 1.9111.94017 2.8808.91249.383-.0382.6915-.30329.9862-.52747.2692-.21713.8525-.82979.8525-.82979z"></path><path id="locationMap-AL" d="m376.4 32.62c-.6332.90263-2.6105.60087-2.8515 1.7715-.1936 1.6279.1334 3.2883-.2031 4.9082-.1783.57985.041 2.0527-.334 2.3438 1.1588 1.0561.059 2.859.092 4.0781-.055 1.5036-.2298 3.0039-.2793 4.5039.4191 1.429-1.0962 2.6309-.461 3.957.7205 1.0298 1.6356 1.9134 2.3282 2.9707 1.0848 1.4521 2.5024 3.4689 3.625 4.5918.6524.19993.877-1.4052 1.4765-1.8535 1.6757-1.3149 3.3121-2.883 4.4414-4.6758.5987-1.0732 1.3745-2.5946 1.6289-3.7695-.8624-.7573-.411-1.7054-.9062-2.1836-1.5832-2.1723-2.6069-5.5599-2.9805-8.1523-.1603-.99218-.9541-1.7393-1.2129-2.7012-.3835-2.0935-2.0049-3.5923-3.5234-4.9355-.7179-.61322-.2685-.66906-.8399-.85352z"></path><path id="locationMap-BA" d="m367.4 7.01c-.1786.26631-1.1235.27527-1.8359.32227-2.3712.22503-5.0992-.898-7.1543.74414-1.5673 1.4473-3.7235.5829-5.5528.32617-2.4358-.68387-5.0723-1.2967-7.5332-.3457-2.3885.65638-2.4856 3.7102-1.8027 5.6445 1.0193 1.896 3.0358 3.0029 4.4258 4.6133 1.4284 1.6458 2.711 3.4682 4.6582 4.5625 1.8929 1.8197 2.8121 4.5138 3.1562 7.0644.2141.79529.5609 1.2934.3262 1.3379.051.50702.9041 1.2195 1.207 1.666.4453.99644 1.0782 1.9554 2.1875 2.2793 1.1168.88469 2.3681.71494 3.2266-.39258.6138-.0486 1.4752-.0935 1.6836-.47266.4141-.7539-.2272-2.1861.1641-3.0957.4972-2.0018 1.5279-3.913 3.1113-5.2578.785-1.0829 1.9539-1.8675 2.6894-2.9805.2199-.33271-.1937-1.1282.078-1.4258.4142-1.4371-.4971-2.8342-1.0977-4.0859-.7956-1.2631-1.5463-2.6083-1.6289-4.1367-.1456-1.1427-.2999-2.5417.6016-3.4004.5841-.95678-.2515-2.4235-.9102-2.9668z"></path><path id="locationMap-BG" d="m393.8 15.77c.4902.83212-.3764 1.8506-.2578 3.0293.2476 2.6571 2.83 3.8947 4.1386 5.9277 1.0465 1.6226.038 3.4939-1.0449 4.7812-.7937 1.0724-1.5818 2.6968-1.5918 4.0977.7497.63987.4417 2.314 1.4102 2.7988 1.2958.89385 2.3039 3.2142 2.4648 4.7793.2949 1.6825.1733 3.2996.3887 4.9805.7542-.74668 2.0615-.8213 2.9121-1.3848 3.0846.30032 6.227.20301 9.291-.88281 3.1179.25929 6.1618 4.0544 9.1914 1.2207 3.0229-1.5506 3.9343-6.1886 7.9688-6.0312.9253-.12846 1.3111-1.1335 2.1484-1.8066 1.2372-.87501 2.6839-.37861 3.8379-1.3613-.1043-1.9788-.229-5.8108.3574-7.7188.4701-1.7739 1.7278-3.0654 3.3203-3.5586 1.1362-1.3712.013-3.6588.2364-5.0195-1.0082.41885-3.3585-.9313-4.4824-1.0527-2.8694-.36313-5.3333-2.9392-8.3165-2.2344-2.8347.369-4.9728 2.3835-6.6269 4.5625-1.4635 3.1738-4.7116 2.4611-7.4258 1.6836-2.6943-.89993-5.5095.0802-8.1465-.14062-2.2724-2.1406-4.4176-4.4836-7.4648-5.5293-.6924-.75185-1.4717-.59304-2.3086-1.1406z"></path><path id="locationMap-CY" d="m484.9 108.08c1.2494-.33278 2.52-.93412 2.7066-2.3546.9195-1.3021 2.7348-.91043 3.9908-1.6099-4.0767-3.0649-9.0305 1.5179-10.866-1.8861-1.2779-.44023-3.255.58966-2.8508 2.1053.4731 1.518.9253 3.2519 2.4945 4.0104 1.4411.82214 3.0752.12656 4.5248-.26506z"></path><path id="locationMap-ES" d="m229.98 34.398c-.3948-.0151-.7919.009-1.1915.0898-1.2512.12096-2.5082.15776-3.7578.10938a384.4 384.4.0 0 0-54.939 30.689c-.2004 1.3091-.1422 2.631.4101 3.9883-.7747 4.8119.981 9.9323-.5586 14.049s-1.1607 4.1667-1.7109 7.6699c.3282.75409 2.5941 1.2727 4.6934 1.5566 3.7836 4.6898 6.9574 10.83 13.676 7.9258 5.1145-3.3257 11.696-5.5854 18.051-5.125 7.9426 1.3967 7.5074-8.6839 14.549-9.7754-.8786-6.7392 12.026-8.8078 4.7011-15.801-1.1289-7.6382 5.3734-13.099 10.613-17.178 7.0966-1.1743 15.822-4.9723 15.785-13.322-.762-.64997-1.7461-1.4334-2.6075-1.6934-2.2471.13211-4.0555.58618-6.9179-.96484-.1551-.0554-.305-.0837-.4531-.0977-.02-.001-.044-.003-.063-.004-.024-.002-.045 92e-5-.068.0-.4384.0153-.6589.25639-1.3281.002.0.0-.5824.61295-.8516.83008-.2947.22418-.6033.48914-.9863.52734-.9697.0277-2.0531-.19501-2.8809-.91211-.3552-.56545-.5106-1.3086-.6777-1.9551-1.1393-.17608-2.3021-.564-3.4863-.60937zm25.572 26.309c-.208-.0173-.5442.21672-1.0527.86914.4382 4.0069 1.9541-.79412 1.0527-.86914zm-7.9082 2.9453c-.3515-.0435-.6965.0857-1.0449.52343-2.9346 6.8651 5.4067 6.3956 4.6445.88281-1.4334.41086-2.5451-1.2758-3.5996-1.4062zm-11.26 10.5c-.2574.0248-.5533.15254-.8847.42382 2.6269 4.2567 2.687-.59767.8847-.42382zm-119.79 89.301c-.1137.0354-.2744.19376-.4942.52343-.2234 4.9917 1.2903-.77149.4942-.52343zm-33.488 3.5195c-.2264-.0262-.4945-.0267-.8066.004-1.4994 6.5328 4.2022.39004.8066-.004zm30.813 2.2715c-.3561.01-.9389.19694-1.8184.63281-4.305 7.8818 4.3106-.70037 1.8184-.63281zm-22.139.69922-.6797.63281c-4.8185 5.1309 4.0422 4.7983 1.3868-.5625zm7.42 3.5332c-2.004 8.2429 3.9021 2.3287.0.0z"></path><path id="locationMap-FR" d="m275.89 15.634a384.4 384.4.0 0 0-50.869 18.967c1.2398.0466 2.4871.009 3.7285-.11132 1.5981-.32515 3.1586.28475 4.6777.51953.6682.0884 2.1937-.30263 2.8203-.0664.5228.2436.9452.58999 1.4922.77149.4782.19303 1.084.80468 1.084.80468.9085.34557.9859-.23122 1.9121.0996 2.8624 1.551 4.6709 1.097 6.918.96485.347-2.0852 1.2367-4.7402 2.1152-6.5566 4.1757-3.9662 10.108-3.5195 14.33-.13672 4.0564 2.4671 9.8058 4.3582 12.963-.51172 2.1778-2.9816 5.8192-1.8038 5.7305-6.6152-3.0064-2.7833-6.6631-1.6151-6.0508-5.5664-.4918-.72756-.7512-1.604-.8516-2.5625zm18.041 18.939c-3.8002.21295-2.5033 3.7773-3.9687 6.3047-1.3537 3.2321 4.264 10.606 4.4863 3.8867 2.3243-2.589 1.2387-7.4466.3125-10.188-.301-.0169-.5767-.0176-.8301-.004z"></path><path id="locationMap-GR" d="m428.7 39.09c-4.0345-.1574-4.9459 4.4806-7.9688 6.0312-3.0296 2.8338-6.0735-.96141-9.1914-1.2207-3.064 1.0858-6.2064 1.1831-9.291.88281-.8506.56346-2.1579.63808-2.9121 1.3848-.4055 1.0172-1.2486.70352-1.8574 1.3281-1.1509 1.0384-2.791.91955-4.2285.87695-1.4151-.10086-2.8325.22223-4.084.88477-.9414.381-2.2163 1.8152-3.2481 2.1973-.2544 1.175-1.0302 2.6964-1.6289 3.7695-1.1293 1.7927-2.7657 3.3609-4.4414 4.6758-.5995.44834-.8241 2.0534-1.4765 1.8535 1.4115 1.813 3.0231 5.3161 4.6211 6.4141 2.2343 2.2122 5.2322 3.5982 5.0878 7.1172 2.3306-.17647 5.1487-1.7424 7.5254.4375 2.9614 2.1108 6.6689-1.609 9.2129 1.5977-.7142 3.028 4.2161 5.298 5.3438 1.9297-.7084-3.1265-3.0278-5.2687-5.3164-7.2539-1.0511-2.9836-1.7578-6.7025-4.209-9.0332-3.672-1.2412-3.6123-4.5776-3.416-7.7774 1.1044-4.2436 5.0448.90988 4.7226 3.4512 1.6709 2.8172 7.2431.13159 6.1055-2.5254-1.258-.5922-5.1294-2.3666-1.8223-3.1875 2.7881-3.6221 5.5711.60989 8.4805.78907 1.2223-1.2626 2.7914-4.3296 4.3418-1.4297.9097 1.2246 3.7828 2.3362 4.5918 1.3867-.1384-1.0779.9424-2.0453 1.5078-2.873 2.3386-3.1584 1.3777-8.5761 3.5508-11.707zm-21.066 30.803c-.1748.0353-.2803.18321-.2852.49219 1.368 2.4758 3.6795 4.5222 6.7012 4.6992 2.4979-.65135-2.6742-2.0276-2.7754-3.625-.8227.21045-2.8833-1.7195-3.6406-1.5664zm-13.76 8.0664c-.2606-.006-.5233-.004-.7851.008-1.7459.0793-3.492.55085-4.8575 1.2539-1.7394 3.143 3.8018 4.2841 2.1817 7.7148.2842 2.6467 2.3092 4.2563 4.8359 2.8477 3.5343-1.1765 1.6572 5.7027 4.9863 2.7012 1.2787.10845 1.2854 5.1372 2.8145 1.9453.8629-2.9324 1.7238.65047 2.4316.93164 1.8502-3.0291-3.0845-5.6662-2.8066-8.9902 1.9449-1.9884 2.0028-5.1832-.4649-7.248-1.9573-.55021-4.05-.56279-6.0605-.83984-.7237-.20329-1.4935-.30544-2.2754-.32422zm49.092 17.258c-.4817 75e-5-.9631.11157-1.416.26758-3.733 2.8387 5.8406 2.2074 1.8965-.22851-.1597-.0263-.3199-.0393-.4805-.0391zm-7.5625 5.7305c-.2393-.01-.5222.001-.8535.0371-1.7281 4.8919 4.4432.10695.8535-.0371zm-22.689 1.3027c-1.2632.0578-2.4634.52325-3.3653 1.5332.2072 1.5945 2.8157 1.5348 4 2.1016 3.4554-.47734 4.0361 5.8716 7.4356 2.8672 2.7429-2.4473 6.6987-.53979 9.8906-1.8047 2.4474-3.6298-3.7632-4.8503-5.3379-1.7441-2.3103-.43929-4.5033-1.5084-6.8555-.47266-1.3803-1.5385-3.6622-2.5768-5.7675-2.4805z"></path><path id="locationMap-HR" d="m364.6.75931c-2.5773 1.4908-6.131 1.149-8.8066.33593-.025-.007-.046-.0164-.07-.0234a384.4 384.4.0 0 0-17.342 1.6934c-1.2122.90726-2.7847 1.1713-4.289 1.3027-1.8478.53558-4.1459-.77266-5.0918 1.2109-.643.28748-2.2524-.62937-2.8301-.39063.3649.49988.2233 1.3314.5957 1.916.6094 1.6711-2.7412 2.1096-.9531 3.7559 1.7108 1.2189 2.225 3.6283 3.7871 4.7617 1.9897.32557 3.3171-.90275 2.8789-2.9141.212-2.1245 2.7861-2.2157 3.3789-.26562 1.4208 1.5001 2.9982 2.9938 2.2949 5.2539.3313 1.7721.4072 3.7212 1.1465 5.3457 1.6239 1.4136 1.464 4.4991 3.8848 5.0391 1.7428.4863 3.8882-.61578 5.3203.80078 1.3931 1.1539 3.1114 2.1582 4.9902 1.8945.7734-.0475 2.1575.88545 2.584.80469.2347-.0445-.1121-.54065-.3262-1.3359-.3441-2.5506-1.2633-5.2448-3.1562-7.0644-1.9472-1.0944-3.2298-2.9186-4.6582-4.5644-1.39-1.6103-3.4065-2.7173-4.4258-4.6133-.6829-1.9343-.5858-4.9862 1.8027-5.6426 2.4609-.95102 5.0974-.34012 7.5332.34375 1.8293.25673 3.9855 1.1211 5.5528-.32617 2.0551-1.6421 4.7831-.51911 7.1543-.74414.7124-.047 1.6573-.056 1.8359-.32227.5764-.85936-1.6513-2.7696-2.1992-3.6894-.3319-.71414-.2277-1.8634-.5918-2.5625z"></path><path id="locationMap-HU" d="m364.99.49173a384.4 384.4.0 0 0-9.3086.58203c.023.007.043.0172.066.0234 2.6756.813 6.2294 1.1569 8.8067-.33399.1388-.0836.2948-.18453.4355-.27148z"></path><path id="locationMap-IT" d="m326.29 4.4175a384.4 384.4.0 0 0-.7032.10742c.2099.10623.4075.22589.584.36524.044-.149.078-.31707.1192-.47266zm-8.5039 1.3984a384.4 384.4.0 0 0-41.856 9.8047c.099.96533.3568 1.8482.8515 2.5801-.6123 3.9513 3.0444 2.7831 6.0508 5.5664 3.7626-3.2733 14.302-5.9934 16.816-1.127 4.0583 2.5596 3.8284 6.7331 5.4043 11.107 3.7652 2.4509 5.7686 7.1392 9.6309 9.1797 5.4453-1.199 5.872 5.0482 9.3183 7.1543 3.8753 1.9327 8.6524 2.1531 11.113 6.5664 1.9854 3.9885 8.2941 3.4311 10.387 7.3711 1.7964 3.314 4.1031 7.9984.2872 10.811-.053 2.6742-4.5036 7.6462 1.3847 6.4727 1.9558-2.5226 5.5821-3.0054 7.2285-5.6602-2.9255-4.2727 6.3681-5.7946 1.336-9.2539-1.5053-2.334-7.3651-3.8791-2.8457-6.543 1.2309-2.9877 3.4797-4.8367 5.5469-1.291 2.1202 1.095 7.0152 4.525 5.4296-.81445-2.354-4.3896-7.1832-5.1601-10.92-7.8887-3.5165 1.0783-6.5807-1.57-5.4258-5.2344-3.7802-2.5243-9.4272-.14092-12.422-4.1367-2.2015-4.4716-5.0118-8.3305-7.2813-12.787-5.8569-2.9106-11.823-10.404-9.7012-15.559-2.4423-2.684-1.9527-4.8818-.3339-6.3184zm-22.352 46.34c-2.2277-.12872-4.8956 2.5111-7.2324 3.3262.2735 1.4612.999 2.9203.2011 4.4004-1.1042 4.4543 3.5886 9.5551-.3007 13 3.151 1.8938 7.5758-.89277 10.607-1.9336 1.7577-2.2127-1.8749-4.7671.4101-7.1133-4.7201-2.718 3.2432-5.6953-.9922-9.3574-.7584-1.6337-1.6807-2.2638-2.6933-2.3223zm43.81 25.438c-2.1712.30155-5.4244 3.9448-7.9746 2.0273-4.0009 1.3985-8.8207-.0428-12.168 2.3613 2.7479 4.8649 8.015 7.3469 15.24 7.4531.6694 2.6078 7.5482 5.4262 7.3203.66992-.6047-3.5692-3.404-6.8867-.2949-10.016-.4025-1.432-.8398-2.4513-2.1231-2.4961z"></path><path id="locationMap-ME" d="m370.3 23.03c-.7355 1.113-1.9044 1.8976-2.6894 2.9805-1.5834 1.3448-2.6141 3.256-3.1113 5.2578-.3913.90964.25 2.3418-.1641 3.0957.7017.69845 1.6861 1.8468 2.5488 2.3848.9296.58948 1.9673 1.0444 2.7129 1.8828.9732.92984 2.3715 2.1234 3.3731 3.0176.3747-.29102.1557-1.7639.334-2.3438.3365-1.6199.01-3.2803.2031-4.9082.241-1.1706 2.2183-.86886 2.8515-1.7715.2266-.32304.7634-.9992.9512-1.3809-.4005-.90359-2.7162-3.2199-3.0156-4.1582-.301-1.0907-1.1132-2.0054-2.166-2.4219-.7365-.34762-1.0553-1.3798-1.8282-1.6348z"></path><path id="locationMap-MK" d="m382 41.11c.3736 2.5924 1.3974 5.9804 2.9806 8.1528.4952.47816.043 1.4259.9054 2.1832 1.0318-.38205 2.3082-1.8153 3.2496-2.1963 1.2515-.66254 2.669-.98682 4.0841-.88596 1.4375.0426 3.0769.16258 4.2278-.87587.6088-.62461 1.4509-.31104 1.8564-1.3283-.2154-1.6809-.092-3.2973-.3869-4.9798-.1609-1.5651-1.1704-3.8852-2.4662-4.7791-.9685-.48481-.6591-2.1598-1.4088-2.7996-.8877-.28352-3.1565.1653-4.0398.33999-1.0219.26995-1.1617 1.3866-1.9777 2.0252-1.7233 1.1588-3.5827 2.216-4.8656 3.8953-.5381.61795-1.6679.59252-2.1589 1.2484z"></path><path id="locationMap-MT" d="m331.6 96.98c.3457.45925.8162.80928 1.304 1.105.3417.19182.7065.38749 1.1065.40255.2094-.0174.397-.13535.5817-.2279.2764-.16345.58-.38263.6417-.71947.043-.29936-.1887-.54046-.4006-.71513-.4772-.35799-1.0372-.58835-1.6011-.77199-.4512-.1338-.9253-.25469-1.3989-.19187-.2117.0245-.4432.16204-.4577.3956-.02.25843.1035.50345.2244.72317z"></path><path id="locationMap-PT" d="m170.13 65.259a384.4 384.4.0 0 0-18.002 12.854c-.031 3.8942 3.8928 1.7611 3.9062 5.0684-.2629 3.4721-2.2184 12.359 4.1993 9.5996 2.7567.0731 6.4233-.50505 8.0312-1.7812.5502-3.5032.1713-3.5556 1.7109-7.6719s-.2161-9.2346.5586-14.047c-.5572-1.3693-.6112-2.7009-.4043-4.0215z"></path><path id="locationMap-RO" d="m384-45901e-8a384.4 384.4.0 0 0-2.2637.01c.5581 1.4441.909 3.0003 1.1504 4.4805.051 2.4927 1.3586 4.6884 2.7695 6.6562 1.1258 2.2259 3.7862 2.6534 5.8985 3.4766.8554.11441 2.005.68559 2.2754 1.1445.8369.54759 1.6162.38685 2.3086 1.1387 3.0472 1.0457 5.1924 3.3887 7.4648 5.5293 2.637.22085 5.4522-.75931 8.1465.14062 2.7142.77751 5.9623 1.4902 7.4258-1.6836 1.6541-2.179 3.7922-4.1935 6.6269-4.5625 2.9832-.7048 5.4472 1.8712 8.3164 2.2344 1.124.12144 3.4743 1.4716 4.4825 1.0527.1988-.8223.6309-1.4341.047-2.6309-1.1001-.75863-3.5605-1.2411-1.3476-2.7109 2.283-2.1919 6.7338-.2211 7.2383-4.5039.8299-1.2851.7916-3.1867.4628-4.9629a384.4 384.4.0 0 0-60.602-4.8086 384.4 384.4.0 0 0-.4004.0z"></path><path id="locationMap-RS" d="m381.67.009311a384.4 384.4.0 0 0-16.644.47851c-.1423.088-.2991.19082-.4395.27539.3641.69907.2599 1.8484.5918 2.5625.5479.91985 2.7756 2.8302 2.1992 3.6895.6587.54331 1.4943 2.01.9102 2.9668-.9015.85866-.7472 2.2577-.6016 3.4004.083 1.5284.8333 2.8736 1.629 4.1367.6005 1.2518 1.5118 2.6489 1.0976 4.0859-.2719.29755.1414 1.0931-.078 1.4258.7729.25501 1.0916 1.2871 1.8281 1.6348 1.0528.41651 1.865 1.3312 2.166 2.4219.2994.93828 2.6151 3.2546 3.0156 4.1582 2.1303-.93056 2.3677-2.2475 3.9727-2.9141 1.3959-.87668.9429-2.9202 2.541-2.2461 2.5326-.12486 6.2559 3.8834 6.877 5.998.036.66899-.119 1.6511.2363 1.8633.8833-.17469 3.1514-.62337 4.0391-.33985.01-1.4009.7981-3.0252 1.5918-4.0976 1.0829-1.2873 2.0914-3.1586 1.0449-4.7812-1.3086-2.033-3.8911-3.2706-4.1387-5.9277-.1186-1.1787.748-2.1972.2578-3.0293-.2704-.45891-1.42-1.0281-2.2754-1.1426-2.1123-.82323-4.7726-1.2526-5.8984-3.4785-1.4109-1.9678-2.7181-4.1616-2.7695-6.6543-.2418-1.482-.5931-3.0407-1.1524-4.4863z"></path><path id="locationMap-SI" d="m338.45 2.7574a384.4 384.4.0 0 0-12.108 1.6524c-.042.15887-.076.3304-.1211.48242.5777-.23874 2.1871.67615 2.8301.38867.9459-1.9836 3.244-.67339 5.0918-1.209 1.5107-.132 3.0924-.39806 4.3067-1.3145z"></path><path id="locationMap-TR" d="m485.7 34.48c-.9189-.0246-1.8948.14008-2.9609.5918-4.3725 2.785-9.129 5.0568-13.141 8.2598-5.3973-.65765-8.8412 3.0594-13.736 3.875-3.6995 1.0439-9.2287-1.9724-11.701 1.2383.5163 2.0563 8.4351 2.0028 3.7383 3.7246-5.3092-1.8452-3.7272 6.1989-8.9453 3.2754-5.4557-.29008-12.6-.99664-15.066 5.3359-2.3049 4.0082 1.0147 5.3089 4.4121 4.5254-.042 1.7192-4.5054 5.1242.2754 4.5371 5.862 2.9573-7.4907 6.7314 1.2871 8.7715 4.0067-.3203 4.5799 3.4881 3.6895 6.0293 1.1079 2.2275 10.692 3.7806 3.7343 4.0371-5.2773 1.0423 3.5104 6.1835 5.8438 3.2871 5.2695.68508 10.241 6.5935 15.857 2.7129 3.6466-1.2521 2.113-7.9152 7.543-5.6738 6.1662-2.232 8.9021 5.992 15.156 5.3184 5.6744.43742 11.039-2.8682 15.701-5.3438 3.8297 1.9559 8.0004-6.9921 9.0117-.51367.1195 2.0861-4.0947 5.9335-3.9121 7.2129 2.639.31587 6.2433.5768 8.2598-1.8476 1.7158-2.1034-1.3569-4.5698.3398-6.5898 1.9758-2.6061 4.0552 1.4007 6.4004.4375 2.8145-.22317 3.9353-3.2345 6.3555-4.1836 1.9524-.58727 3.8429.80138 5.8437.48437 3.0308.42847 5.9969-.74832 8.3262-2.6387 2.721-1.82 6.0905-1.2733 9.1602-1.2168 2.2324-.2288 4.9976-1.1454 7.2461-.71875 1.8606.60826 4.7612 1.0426 7.0742.0527 4.1811-.88052 9.2538 1.1087 13.389-.70508.2888-1.2016-.061-2.6571-1.623-4.6641-6.5973-2.4504-3.4376-10.286-4.9394-15.549-.4847-1.1514-.8829-2.6975-1.4434-3.8301-.5525-.43555-.525-1.9318-.7852-2.5703-1.156-1.7746-2.4627-3.4677-3.8125-5.1055-1.1177-1.4364-2.5401-2.5676-3.9824-3.6562-.5496-.24574-1.5687-1.1152-1.5547-1.5039-.4652-.6515-1.0065-1.2712-1.582-1.8672a384.4 384.4.0 0 0-9.582-4.5859c-.079-47e-5-.1575-.007-.2364-.006-1.5284-.52033-3.043 1.2358-4.6933.58984-.8147 4.5796-6.8168 6.2578-10.811 6.5371-6.2854-.59239-12.441.45996-18.674.3125-5.2765.71949-10.66-.67446-13.977-5.0762-4.5885.48795-7.5029-3.1944-11.484-3.3008zm-51.076 1.4394c-1.154.98272-2.6007.48635-3.8379 1.3613-.8373.67314-1.2231 1.6782-2.1484 1.8066-2.1731 3.131-1.2122 8.5486-3.5508 11.707-.5654.82775-1.6462 1.7952-1.5078 2.873 4.7004 1.5832 9.1108-.52284 10.924-5.1621 2.9019-.26295 13.016 2.5467 6.0957-3.8359-3.7469-.94954-4.0403-7.9708-5.9746-8.75zm-15.144 24.963c-.1651.015-.3605.0469-.5898.0996-1.2705 3.3326 3.0664-.32401.5898-.0996zm74.363 36.615c-1.1791.41453-2.2118 1.3174-3.0762 2.2246-1.0062 1.3631-2.8569.7735-4.2695.66015-1.1166-.19349-3.0467-.17719-3.0469 1.3828.5396 1.9251-1.8641.96372-2.7187.46484 1.8354 3.404 6.7905-1.1802 10.867 1.8848.7821-1.0049-1.6193-2.4667.25-3.2695.8535-.73268 3.4609-2.2944 1.9942-3.3476z"></path><path id="locationMap-UA" d="m471.05 9.8921c.6383 1.9865 3.0596 2.9684 5.9257 1.8711.2468-.0696.484-.16986.7208-.26953a384.4 384.4.0 0 0-6.6465-1.6016z"></path><path id="locationMap-XK" d="m383.4 25.95c-1.003-.0485-.8079 1.6118-2.0293 2.3789-1.605.66654-1.8424 1.9835-3.9727 2.9141-.1878.38166-.7246 1.0578-.9512 1.3809.5714.18446.122.2403.8399.85352 1.5185 1.3432 3.1399 2.842 3.5234 4.9355.2588.96187 1.0526 1.709 1.2129 2.7012.491-.65587 1.6201-.63205 2.1582-1.25 1.2829-1.6793 3.1419-2.7358 4.8652-3.8945.8161-.63864.9567-1.7554 1.9786-2.0254-.3553-.21221-.2004-1.1943-.2364-1.8633-.6211-2.1147-4.3443-6.1229-6.8769-5.998-.1998-.0843-.3685-.12589-.5117-.13282z"></path><path id="locationMap-AE" d="m674.9 213.78s-2.7749-6.9806-.9657-9.541c1.101-1.8726 1.9249-4.0634 1.4469-6.259-.2357-2.037-.5798-4.2631.5656-6.1016.3658-1.4236 1.3824-2.8357 3.036-2.6546.9253-.144.9198.641 1.9727.7362-.1613-1.6707-.088-5.0808-.8255-6.6765-.4134-1.4144-2.7661.4318-3.9863-.091-1.2503-.6358-1.4036 1.5886-2.9516 1.3063-2.1484.3873-2.0714 3.1595-3.747 4.1939-1.0383 1.4648-1.7822 3.156-3.1264 4.4248-1.4566.8475-2.2906 2.0265-2.5078 3.6906-.6323 1.7135-2.3909 1.9361-3.9297 2.0719-1.399.4513-2.8372-.6057-4.3701-.126-1.9363-.9068-3.9297-.1294-5.5424 1.001-3.5618 1.1336-5.7132-.017-7.7717-.6043-1.1933 1.965 1.7452 5.2169 2.9248 6.4074.9557 1.3741 1.403 3.4437 3.3104 3.8112 2.2908.6522 4.6667-.2989 6.9578.1636 2.2276.7282 4.3324 1.9488 6.7256 2.0549 3.9369.5217 12.784 2.1922 12.784 2.1922z"></path><path id="locationMap-AM" d="m557.6 41.232c-.2777.19963-.5481.41192-.8086.64063-.014.38868 1.0051 1.2582 1.5547 1.5039 1.4423 1.0886 2.8647 2.2198 3.9824 3.6562 1.3498 1.6378 2.6565 3.3308 3.8125 5.1055.2602.63855.2327 2.1348.7852 2.5703.2768.2182.9144-.23988 1.375-.38282 2.038.1554 4.195.81639 5.539 2.4453 1.4331.60852 1.3572 2.1572 1.75 3.707 5.047 2.3259 5.6047-1.4824 8.8711-4.1445-.1032-.13223-.2245-.24948-.3379-.37305a384.4 384.4.0 0 0-26.523-14.729z"></path><path id="locationMap-AZ" d="m568.24 54.33c-.4606.14294-1.0982.60101-1.375.38281.5605 1.1326.9587 2.6786 1.4433 3.8301 1.9553 1.0587 4.8777 1.8938 7.2207 1.9394-.3928-1.5498-.3169-3.0985-1.75-3.707-1.344-1.6289-3.501-2.2899-5.539-2.4453zm.068 4.2129c.076.28316.1374.57069.1972.85938-.055-.29159-.1163-.58101-.1953-.85938zm15.664-2.6758c.1466.1534.2971.30336.4277.4707.1078.0543.2162.0862.3262.10742.029.006.06.006.09.01.043.005.087.004.1309.006a384.4 384.4.0 0 0-.9746-.59375z"></path><path id="locationMap-BH" d="m632.6 179.25c-.075.50349-.5412.62078-.4536 1.1472.034.2809.4043.38922.4708.659.1451.35785.212.90926.4475 1.5008.4016.2833.8632 1.1708 1.4641 1.1073.2961-.84055.2683-4.8515.3947-5.7248.016-.4808-.505-.32136-1.0532-.0958-.5413.44955-1.2236.041-1.2703 1.4064z"></path><path id="locationMap-GE" d="m540.32 33.043c.397.93061.5608 1.9437.3457 2.9629 1.6503.64591 3.165-1.1121 4.6934-.5918.07-.001.1391.006.209.006a384.4 384.4.0 0 0-5.2481-2.377zm14.879 6.9883c.5666.58901 1.1016 1.199 1.5606 1.8418.2651-.23283.5412-.4477.8242-.65039a384.4 384.4.0 0 0-2.3848-1.1914z"></path><path id="locationMap-IL" d="m506.6 115.71c-1.7645.26066-2.7544 1.8583-4.043 2.9473-.9808.86379-2.2739 1.235-3.5 1.5234-.4424.19935-.082 1.2278-.5781 2.1094-.5882 1.5419-1.1502 3.0958-1.3828 4.7402-.274 2.2191-2.2598 8.7461-2.2598 8.7461l8.2852 15.484c.3058-2.3414.8869-8.3034 1.3222-10.74-1.5619-1.7358-1.6135-1.5154-1.5625-3.3926-.062-1.6892-.017-3.3815-.166-5.0664-.094-1.616.8595-2.7924 2.2735-3.4219.5762-.41881 1.391-.11114 2.0546-.76367.2677-.26327.2477-.8591.584-1.4492-.4024-1.8357-1.0094-3.9396-.7324-5.8438.3102-1.0411-.193-2.1072.668-2.7812.265-.36254-1.0714-1.6581-.9629-2.0918z"></path><path id="locationMap-IQ" d="m574.9 78.75c-4.1345 1.8138-9.2078-.17544-13.389.70508-2.313.98981-5.2136.55552-7.0742-.0527.774 1.7043-.035 5.3839-2.289 6.1602-2.0666 1.4385-4.4503 3.5019-4.0078 6.3106.3525 2.5889.9663 5.1731.539 7.8047-.5526 2.9916-.212 6.2767-1.7344 9.0039-2.062 2.3334-5.6275 2.5127-7.7402 4.7949-2.0393 1.4727-4.664 2.3418-6.3535 4.459-.6131.71826-1.2975 1.8545-2.0723 2.3379 2.2384 3.1681 3.4679 6.253 4.8614 9.25 1.4962 1.2546 4.0883.79973 6.1503.66796 3.7574.26048 8.1379-.41426 11.061 2.5332 4.744 3.5051 9.8159 6.6336 14.037 10.775 4.1726 3.1096 8.7122 5.6968 12.824 8.8789 3.2918 2.4434 7.149.38965 10.768.57227 3.1808.16147 8.4093.70038 10.606-.57227-.6631-3.9161 2.4819-9.9666 6.7324-7.2754 2.4711 1.5335 4.565 3.2059 5.5156 4.6113 1.9572.2508 2.7417-.91741 3.3613-1.7266-.6266-1.774-4.2712-4.5428-4.8886-5.1113-3.5916-2.906-6.9131-6.5179-7-11.451-1.1722-3.2255-4.3649-5.5185-6.17-8.5723-3.582-.55946-7.9893-1.9151-8.3554-6.252-1.2565-4.5454-7.3557-6.0897-6.7754-11.426-.227-3.9643 4.2801-6.9312 3.2305-11.109-2.8374-2.327-7.4751-2.9386-8.5059-7.1523-2.1347-1.9007-1.5928-7.4885-3.3301-8.1641z"></path><path id="locationMap-IR" d="m584.43 56.337c-3.2664 2.6621-3.8241 6.4704-8.8711 4.1445-2.343-.0456-5.2654-.88083-7.2207-1.9394 1.5018 5.2633-1.6579 13.098 4.9394 15.549 1.5619 2.0069 1.9118 3.4624 1.6231 4.6641 1.7373.67552 1.1954 6.2634 3.3301 8.1641 1.0308 4.2138 5.6684 4.8254 8.5058 7.1524 1.0496 4.1782-3.4574 7.1434-3.2304 11.107-.5803 5.3361 5.5188 6.8821 6.7753 11.428.3661 4.3369 4.7735 5.6925 8.3555 6.252 1.8052 3.0538 4.9977 5.3467 6.1699 8.5723.087 4.9333 3.4084 8.5453 7 11.451.6174.56851 4.2621 3.3373 4.8887 5.1113 1.5966-1.0696 3.3693-1.3828 2.6289-3.0859 4.2744-3.2408 1.7063 4.1117 6.8184 2.3613 3.9002 4.5272 3.7379 11.346 9.3633 13.713.8452 4.8328 5.4577 7.5982 9.4746 4.4746 5.852 2.9498 11.911 6.7024 16.975 10.447 6.6755.6792 10.74-5.8908 16.951-5.625 8.7632-2.2455 3.6465 12.042 11.699 10.701 4.5377 3.8464 10.799.42033 15.225 4.0488 2.2382-.95892 4.6594-.67631 7.1054-.1875a384.4 384.4.0 0 0-127.98-128.37c-.1788-.006-.3517-.0424-.5234-.12891zm90.41 116.87c-.071.009-.1525.0336-.2441.0781-.3628 1.1275-2.8115 2.1353-1.502 3.3613 1.3782.2783 1.7278-1.4052 2.1231-2.3574.1484-.40198.1229-1.1421-.377-1.082z"></path><path id="locationMap-JO" d="m530.8 120.28c-.2681-.0222-1.1427.9846-1.5664 1.3359-2.5985 1.05-5.6228.99754-7.9043 2.8047-2.0638 1.2953-4.0521 3.3849-6.7031 3.0625-1.8979.0506-5.0857-2.2-6.9805-1.0566-.3363.59012-.3163 1.186-.584 1.4492.2744.82714 1.2481 1.8148.6387 2.4688-.879 1.4458-1.1076 3.3614-1.0937 5.0566-.2312 2.3222-.713 3.9234-2.1426 5.1211-.2177 1.2184-.4728 3.3169-.7109 5.4043-.2382 2.0874-.4604 4.1633-.6133 5.334.5603 1.149.227 2.4185-.014 3.623.01.74873.1033 2.4675.4453 3.2891.2008.0894.4402-.45808.8692-.86718 2.241-1.2849 4.6234.80081 6.9511.10156 2.5045-.3215 4.3603-2.1465 5.8262-4.0547 1.1701-1.4379 2.3435-3.4551 4.4707-3.3574 1.4909-.10275 3.3553.28531 4.4648-.96875.8721-1.8253-.089-3.9206.5625-5.8086.2736-2.0001-.4891-3.9353-1.5156-5.6094-.5922-1.5074-.9381-3.4618.059-4.8652 1.6054-1.6656 4.1135-1.8828 6.2988-2.0859 1.0105-.42035 4.2476-.008 4.1036-1.127-1.3935-2.997-2.623-6.0819-4.8614-9.25z"></path><path id="locationMap-KW" d="m605.6 144.44c-3.0393.1665-5.0153 4.7576-4.4765 7.9394.7781 1.3068 2.7169 2.5686 3.7675 3.623.6119.54314 1.058 1.2331 1.4122 1.9629.7903 1.1888 2.1882 1.7627 3.5351 2.0469 1.3248.35598 2.847.0818 4.1288-.24664-.132-.79869-.3821-1.8247-.9218-2.4702-1.1421-1.5368-3.0675-2.4075-3.8808-4.2012-.4101-.73506-2e-4-1.6703.791-1.9121.8695-.34486 2.5366-1.1609 3.416-1.4668-.9506-1.4054-3.0445-3.0779-5.5156-4.6113-.797-.5046-1.5545-.70248-2.2559-.66406zm-4.4765 7.9394c-.01.006-.021.0101-.031.0156.029.068.065.12628.031-.0156z"></path><path id="locationMap-LB" d="m504.3 104.14c.1626 1.1892-.2233 2.5378-.9011 3.5181-.7312 1.323-1.0624 2.8334-1.901 4.1034-1.1481 2.0718-1.1145 4.5317-2.0308 6.6907-.2202.49298-.2341 1.241-.4595 1.7272 1.2261-.28846 2.5203-.65931 3.5011-1.5231 1.2886-1.089 2.2768-2.6878 4.0413-2.9484 1.2651-.15663.674-1.2267 1.5306-2.0586.7454-.49482 1.4306-1.1493 1.8985-1.9235.6136-1.0086.3229-2.2288-.1023-3.2439-.3251-1.0351.5635-1.7305-.4181-2.2005-1.4331-.67676-2.9075-1.2713-4.408-1.7838-.261-.0825-.4834-.29812-.7507-.35752z"></path><path id="locationMap-OM" d="m678.8 176.01c-1.2775-.0656-2.2642 2.0518-2.7363 3.1348-.7736 1.1045-.8196 2.8715-.8594 4.1738.2624-.21835.5501-.33003.961-.12109 1.1765.50411 3.4011-1.187 3.9297-.041.093-2.365.4829-4.8806-.3672-6.7129-.3231-.28734-.6329-.41844-.9278-.43359zm.5223 13.205c-.091-.01-.191-4e-4-.3067.014-1.6536-.1811-2.6713 1.2307-3.0371 2.6543-1.1454 1.8385-.8001 4.0646-.5644 6.1016.478 2.1956-.3463 4.3852-1.4473 6.2578-1.8092 2.5604.9649 9.541.9649 9.541 4.3852 2.5645 4.0098 11.08 2.9668 16.145-2.6345 6.8456-11.377 9.4237-17.713 12.234-2.5153 1.3695-5.1155 2.5222-7.7598 3.5606-8e-4 3e-4.0.0.0.0-3e-4 4e-4 4e-4.0.0.0 2.0627 2.8459 4.81 7.5971 6.6895 10.457 1.6562 2.6129-.9834 6.8225 2.875 8.4356 1.5217.8047 2.0604 2.4069 3.3744 3.4056 3.053-1.8481 7.2302-2.143 10.241-3.9076 2.9748-.606 7.263-1.4902 6.0469-5.6699.1215-2.9882 5.2713-1.3926 6.9883-3.6504 3.7759-.777.1501-6.5516 4.0469-6.793 2.5608-1.185 4.9216-3.1352 6.9277-4.8203-1.3012-3.1172 1.1134-5.9257.8008-9 1.7527-1.5183 5.9301 1.827 5.7676-2.3301-1.2687-3.2326 2.0526-5.7115 3.7109-7.3515 2.6247-1.8325 3.1117-4.5299 2.623-7.6328 1.2836-3.7743-1.6251-7.0772-5.2343-7.7598-2.8737-1.6455-2.1844-6.1634-5.8731-6.9219-4.037-1.4651-8.084-3.1303-12.394-3.6035-3.2206-.9305-6.8351-5.4264-8.0254-8.6191-.9213-.083-1.0328-.695-1.666-.75z"></path><path id="locationMap-PS" d="m507 127.87c-.6636.65253-1.4784.34486-2.0546.76367-1.414.62953-2.3675 1.8059-2.2735 3.4219.1485 1.6849.104 3.3772.166 5.0664-.051 1.8772 6e-4 1.6568 1.5625 3.3926 1.428-1.1975 1.9096-2.7982 2.1407-5.1191-.014-1.6952.2147-3.6108 1.0937-5.0566.6094-.65391-.3643-1.6416-.6387-2.4688z"></path><path id="locationMap-QA" d="m640.1 179.86c-.6244.043-1.2171.5129-1.4414 1.1016-.5649 1.0952-.7362 2.3368-.8457 3.5468-.2974 1.5095-1.5313 2.7589-1.3965 4.3653.036.855 1.0115 1.2453 1.1387 2.0722.3208 1.1065.1993 5.0078.7187 6.836 1.0241.9246 2.2902 1.4633 3.916 1.373.9555-2.4911 1.8963-8.535 2.4024-11.154.1353-1.4509-.3127-2.8821-.9082-4.1875-1.1377-1.1638-1.6863-2.8015-2.9629-3.8437-.203-.091-.413-.1239-.6211-.1094z"></path><path id="locationMap-SA" d="m535.6 129.53c.144 1.1189-3.0931.7066-4.1036 1.127-2.1853.2031-4.6934.42029-6.2988 2.0859-.9959 1.4035-.6504 3.3579-.059 4.8652 1.0265 1.6741 1.7892 3.6093 1.5156 5.6094-.6512 1.888.3096 3.9833-.5625 5.8086-1.1095 1.2541-2.9739.866-4.4648.96875-2.1272-.0977-3.3006 1.9195-4.4707 3.3574-1.4659 1.9082-3.3217 3.7332-5.8262 4.0547-2.3277.69925-4.7101-1.3865-6.9511-.10156-.429.4091-.6684.95658-.8692.86718-.5475 8.7321 2.4694 16.549 8.7539 19.311 2.3753 7.9428 8.0892 13.603 13.551 19.299 1.1311 6.8755 5.6787 9.1609 11.449 11 5.6606 5.4535 6.4649 13.745 8.2637 20.826.1497 7.9192 8.0242 10.005 13.338 12.213 3.879 5.2946 7.6371 10.752 10.201 16.922 3.2871 3.2533 6.2666 11.504 9.496 12.117 2.4554-8.6196 13.504-11.693 20.201-6.4765 6.7582-.9641 10.605 13.073 15.184 2.8281 3.2177-7.3838 10.693-10.928 17.94-13.172 6.7086-2.7755 13.746-4.6688 20.488-7.3164 2.6451-1.0386 5.2458-2.1926 7.7618-3.5625 6.3363-2.8106 15.078-5.3887 17.713-12.234 1.043-5.0644 1.4184-13.58-2.9668-16.145.0.0-8.8463-1.6697-12.783-2.1914-2.3932-.1061-4.499-1.3265-6.7266-2.0547-2.2911-.4625-4.6662.4882-6.957-.164-1.9074-.3675-2.3549-2.4365-3.3106-3.8106-1.1796-1.1905-4.1191-4.4432-2.9258-6.4082-6.4007.3554-7.252-8.9958-9.5332-13.553-6.7715.1987-3.4458-9.8579-9.6621-11.631-5.9756-2.2507-6.0838-10.906-9.0957-14.197-1.2818.32844-2.8041.60208-4.1289.2461-1.3469-.28422-2.7448-.85807-3.5351-2.0469-.3542-.72981-.8003-1.4198-1.4122-1.9629-1.0506-1.0545-2.9894-2.3163-3.7675-3.623.034.14188-9e-4.0836-.031.0156-2.213 1.2512-7.4074.7174-10.574.55664-3.6186-.18262-7.4758 1.8712-10.768-.57227-4.112-3.1821-8.6516-5.7693-12.824-8.8789-4.2212-4.1418-9.2931-7.2703-14.037-10.775-2.9227-2.9475-7.3032-2.2727-11.061-2.5332-2.062.13177-4.6541.58665-6.1503-.66796z"></path><path id="locationMap-SY" d="m552.8 79.3c-1.8628.0702-3.8845.64905-5.5938.82422-3.0697-.0565-6.4391-.60319-9.1601 1.2168-2.3293 1.8903-5.2954 3.0671-8.3262 2.6387-2.0008.31701-3.8913-1.0716-5.8437-.48437-2.4202.94905-3.541 3.9604-6.3555 4.1836-2.3452.96324-4.4246-3.0436-6.4004-.4375-1.6967 2.0201 1.376 4.4864-.3398 6.5898-2.0165 2.4244-5.6208 2.1635-8.2598 1.8477-.9578 2.1439.1001 4.3321 1.207 6.168.1158.74213.2601 1.6079.6348 2.291.2673.0594.489.27493.75.35743 1.5005.51252 2.9751 1.1064 4.4082 1.7832.9816.47.093 1.1661.418 2.2012.4252 1.0151.7151 2.2355.1015 3.2441-.4679.77423-1.153 1.429-1.8984 1.9238-.8566.83191-.2642 1.902-1.5293 2.0586-.1085.4337 1.2279 1.7293.9629 2.0918-.861.67407-.3578 1.7401-.668 2.7812-.277 1.9041.33 4.008.7324 5.8438 1.8948-1.1434 5.0826 1.1072 6.9805 1.0566 2.651.32242 4.6393-1.7672 6.7031-3.0625 2.2815-1.8072 5.3058-1.7547 7.9043-2.8047.4237-.35133 1.2983-1.3581 1.5664-1.3359.7748-.48336 1.4592-1.6196 2.0723-2.3379 1.6895-2.1172 4.3142-2.9863 6.3535-4.459 2.1127-2.2822 5.6782-2.4615 7.7402-4.7949 1.5224-2.7272 1.1818-6.0123 1.7344-9.0039.4273-2.6316-.1865-5.2158-.539-7.8047-.4425-2.8087 1.9412-4.872 4.0078-6.3106 2.2545-.77626 3.063-4.4558 2.289-6.1602-.281-.0533-.5701-.0863-.8652-.10156-.2583-.0134-.521-.0138-.7871-.004z"></path><path id="locationMap-YE" d="m652.4 245.72c-6.7423 2.6476-13.78 4.5409-20.488 7.3164-7.2463 2.2442-14.722 5.7881-17.94 13.172-4.5789 10.245-8.4254-3.7922-15.184-2.8281-6.6967-5.2164-17.746-2.1431-20.201 6.4765 2.4123 3.6615.7689 8.6264-.7168 12.227 1.5257 2.8532 4.6683 5.0144 4.043 8.6758-1.4793 3.7782 1.449 6.0862 3.4766 8.5195 2.5533 2.5181 4.8324 6.2376 9.2246 4.7637 4.6578.6447 6.1889-4.2212 9.4238-5.9258 3.5274-1.2426 7.8042.4123 10.67-2.5332 3.7297-1.262 7.29-3.1246 11.293-3.4043 3.7613-.567 5.5495-4.2136 8.6621-5.7891 3.3508-1.6535 6.6726-3.3952 10.506-3.7343 4.1623.039 6.9051-4.2973 11.221-3.2754 5.3879.6567.525-7.1235 5.1113-8.4766 2.0419-.2744 2.5612-1.8914 3.8379-2.8828 4e-4.0.0.0.0.0-1.314-.9987-1.8533-2.6016-3.375-3.4063-3.8584-1.6131-1.2188-5.8226-2.875-8.4355-1.8791-2.8592-4.625-7.609-6.6875-10.455zm15.795 56.201c-1.5945-.022-3.4898.4501-4.4063.959-.7959.1142-1.5589.3728-2.334.5762-3.0589-.358-3.8433 3.0455-.2656 2.623 3.9195 2.8089 5.6989-2.3122 9.6113-2.1543.2836-1.4665-1.0109-1.982-2.6054-2.0039z"></path><path id="locationMap-KI" d="m1700.3 394.48c-.6497 5.366 6.951.3318.0.0zm-2756.4 44c-.7818 1.8554 2.1449-.4422.0.0zm5.5-4.2c.7017 2.5146 1.5054-2.9402.0.0zm.3 5.6c.5323 2.0956 2.3586-1.6503.0.0z"></path></svg>

</div>

<script>
    setTimeout(function() {{
        var mapElement = document.querySelector('.folium-map');
        var compass = document.getElementById('bussola-nord');
        var minimap = document.getElementById('minimappa-custom');

        if (mapElement && compass) {{ mapElement.appendChild(compass); }}
        if (mapElement && minimap) {{ mapElement.appendChild(minimap); }}
    }}, 200);
</script>
"""

mappa_tour.get_root().html.add_child(folium.Element(html_layout))
mappa_tour.save("mappa_tour_perfetta.html")
print("Mappa HTML salvata con successo!")

Script per lo screenshot .png della mappa, conrisoluzione ottimizzata in 4k creata precedentemente in HTML

In [ ]:
# Assicurati di aver lanciato l'installazione dei font almeno una volta in questa sessione
!apt-get update -q
!apt-get install -y wget curl unzip libxi6 libgconf-2-4 libnss3 fonts-noto-color-emoji -q
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!dpkg -i google-chrome-stable_current_amd64.deb || apt-get -fy install -q
!pip install selenium chromedriver-autoinstaller -q

import time
import os
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import chromedriver_autoinstaller
from google.colab import files

chromedriver_autoinstaller.install()

print("Avvio di Google Chrome (con supporto Emoji e Altissima Risoluzione abilitati)...")

file_html = "mappa_tour_perfetta.html"
percorso_assoluto = "file://" + os.path.abspath(file_html)

chrome_options = Options()
chrome_options.add_argument('--headless=new')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--window-size=1080,1440') # Mantiene le proporzioni del layout perfette
chrome_options.add_argument('--force-device-scale-factor=3') # IL TRUCCO MAGICO: Moltiplica x3 i pixel (Qualità Retina)

try:
    driver = webdriver.Chrome(options=chrome_options)
    driver.get(percorso_assoluto)

    print("Mappa 4K in caricamento... attendiamo 8 secondi per scaricare tutti i dettagli.")
    time.sleep(8) # Qualche secondo in più per dare il tempo a Esri di caricare la mappa in altissima definizione

    nome_immagine = "mappa_tour_perfetta_finale.png"
    driver.save_screenshot(nome_immagine)
    driver.quit()

    print(f"✓ Immagine ad altissima risoluzione '{nome_immagine}' salvata con successo! Avvio del download...")
    files.download(nome_immagine)

except Exception as e:
    print(f"Si è verificato un errore: {e}")